# Неделя 03 — Скрытые пары
(i.sokolov@innopolis.university)

Эта базовая модель предсказывает, относится ли скрытая пара к положительному классу (`target = 1`), используя 512 анонимных числовых признаков.

Мы обучаем одно небольшое дерево решений на 2 520 строках обучающей выборки, оцениваем его на неизменённой валидационной выборке из 560 строк и используем тестовую выборку из 1 116 строк только для получения вероятностей. Это учебная базовая модель, а не решение, оптимизированное для лидерборда.

Во второй части ноутбука (Задание 8) базовая модель сравнивается с одиночными деревьями разной глубины, случайным лесом, Extra Trees и градиентным бустингом, а также с референсными моделями вне класса деревьев (логистическая регрессия, SVM); лучшая референсная оценка затем передаётся деревьям как признак (стекинг). Итоговая модель выбирается по валидационной выборке и записывает файл `submission.csv`. Ноутбук устроен так, что «Restart kernel → Run all» заново воспроизводит все таблицы и отправляемые предсказания.

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

Чтобы использовать Colab, загрузите этот ноутбук вместе с файлами `train.csv`, `validation.csv`, `test.csv` и `sample_submission.csv`, затем запустите все ячейки. После загрузки файлов интернет и GPU не требуются.

## 1. Импорт библиотек и воспроизводимость

pandas используется для работы с CSV-таблицами, NumPy — для числовых проверок, а scikit-learn предоставляет модель и метрику ROC-AUC. PyTorch не используется, поскольку предоставленные файлы уже содержат необходимые 512 числовых признаков.

In [1]:
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier

SEED = 20260916
random.seed(SEED)
np.random.seed(SEED)
notebook_started = time.perf_counter()

## 2. Загрузка предоставленных файлов

Разбиения уже подготовлены. Не создавайте другое случайное разбиение, не объединяйте выборки и не обучайте ничего на тестовых данных.

In [2]:
# DATA_DIR — каталог, содержащий четыре предоставленных CSV-файла.
DATA_DIR = Path(".")

train = pd.read_csv(DATA_DIR/ "train.csv")
validation = pd.read_csv(DATA_DIR / "validation.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("Обучающая выборка:", train.shape)
print("Валидационная выборка:", validation.shape)
print("Тестовая выборка:", test.shape)
print("Образец отправки:", sample_submission.shape)

Обучающая выборка: (2520, 514)
Валидационная выборка: (560, 514)
Тестовая выборка: (1116, 513)
Образец отправки: (1116, 2)


## 3. Проверка входных данных

Перед моделированием проверьте ожидаемое число строк, обязательные столбцы, пропущенные значения, бинарные целевые значения и одинаковый порядок признаков.

In [3]:
assert train.shape == (2520, 514)
assert validation.shape == (560, 514)
assert test.shape == (1116, 513)
assert sample_submission.shape == (1116, 2)

assert {"row_id", "target"}.issubset(train.columns)
assert {"row_id", "target"}.issubset(validation.columns)
assert "row_id" in test.columns and "target" not in test.columns
assert list(sample_submission.columns) == ["row_id", "target"]

assert not train.isna().any().any()
assert not validation.isna().any().any()
assert not test.isna().any().any()
assert set(train["target"].unique()) == {0, 1}
assert set(validation["target"].unique()) == {0, 1}

## 4. Выбор признаков

`row_id` используется для сопоставления строк, а `target` является ответом. Ни один из этих столбцов не является признаком модели. Во всех выборках должны присутствовать одни и те же 512 упорядоченных столбцов признаков.

### TODO(student) — Задание 1: выбрать столбцы признаков

Создайте `feature_columns` из `train.columns`, исключив `row_id` и `target`. Доступные переменные — это столбцы таблицы `train`. Результат представляет собой упорядоченные входные признаки модели и должен быть списком ровно из 512 названий.

In [4]:
feature_columns = [column for column in train.columns if column not in ("row_id", "target")]

In [5]:
assert len(feature_columns) == 512
assert not {"row_id", "target"}.intersection(feature_columns)
assert feature_columns == [column for column in validation.columns if column not in {"row_id", "target"}]
assert feature_columns == [column for column in test.columns if column != "row_id"]

## 5. Формирование входных данных модели

Не используйте валидационные данные при обучении. Дерево получает предоставленные числовые столбцы напрямую, поэтому обучаемого шага предобработки нет.

### TODO(student) — Задание 2: сформировать матрицы и целевые векторы

Используя `train`, `validation`, `test` и `feature_columns`, создайте `X_train`, `y_train`, `X_validation`, `y_validation` и `X_test`. Они представляют три матрицы признаков и два доступных целевых вектора. Каждая матрица признаков должна иметь 512 столбцов, а длина каждого целевого вектора должна соответствовать своей размеченной выборке.

In [6]:
X_train=train[feature_columns].to_numpy()
y_train=train["target"].to_numpy()
X_validation=validation[feature_columns].to_numpy()
y_validation=validation["target"].to_numpy()
X_test=test[feature_columns].to_numpy()

In [7]:
assert X_train.shape == (2520, 512)
assert X_validation.shape == (560, 512)
assert X_test.shape == (1116, 512)
assert len(y_train) == len(X_train)
assert len(y_validation) == len(X_validation)

## 6. Определение и обучение базовой модели

Дерево глубины 3 прозрачно и быстро. `min_samples_leaf=20` не позволяет строить правила по очень маленьким группам. Модель обучается только на строках обучающей выборки.

### TODO(student) — Задание 3: создать модель

Создайте `model` как `DecisionTreeClassifier` с параметрами `max_depth=3`, `min_samples_leaf=20` и `random_state=SEED`. Результатом должен быть ещё не обученный классификатор.

In [8]:
model=DecisionTreeClassifier(max_depth=3,min_samples_leaf=20,random_state=SEED)

### TODO(student) — Задание 4: обучить модель

Обучите `model`, используя только `X_train` и `y_train`. Результатом должен быть классификатор, обученный исключительно на обучающих строках и имеющий метод `predict_proba`.

In [9]:
model.fit(X_train,y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",3
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",20
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",20260916
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of sam

## 7. Оценка на валидационной выборке

ROC-AUC показывает, насколько хорошо непрерывные оценки ранжируют положительные примеры выше отрицательных при всех возможных порогах. Классовые метки после применения одного порога теряют информацию о ранжировании, поэтому их не следует использовать для расчёта ROC-AUC.

### TODO(student) — Задание 5: валидационные вероятности и ROC-AUC

Используя `model`, `X_validation` и `y_validation`, создайте вероятности класса 1 `validation_probability` и скаляр `validation_auc`. На каждую валидационную строку должна приходиться одна вероятность, а AUC должен быть конечным и приблизительно равным 0.7473.

In [10]:
validation_probability=model.predict_proba(X_validation)[: , 1]
validation_auc=roc_auc_score(y_validation,validation_probability)

In [11]:
assert validation_probability.shape == (len(validation),)
assert np.isfinite(validation_auc)
assert np.isclose(validation_auc, 0.7473086734693878)
assert 0.0 <= validation_auc <= 1.0

## 8. Предсказание для тестовой выборки

Тестовая выборка остаётся неразмеченной и используется только после обучения. Повторное предсказание проверяет детерминированность. Ориентировочный публичный результат преподавателя равен примерно 0.72157; публичные метки недоступны этому ноутбуку и не используются в нём.

### TODO(student) — Задание 6: вероятности для тестовой выборки

Используя `model` и `X_test`, создайте вероятности класса 1 `test_probability`. Повторите предсказание и сохраните его как `test_probability_repeat`; оба массива должны быть одинаковыми и содержать по одной вероятности из диапазона `[0, 1]` для каждой тестовой строки.

In [12]:
test_probability = model.predict_proba(X_test)[:, 1]
test_probability_repeat = model.predict_proba(X_test)[:, 1]

In [13]:
assert test_probability.shape == (len(test),)
assert np.array_equal(test_probability, test_probability_repeat)
assert np.all((test_probability >= 0.0) & (test_probability <= 1.0))

## 9. Создание и проверка файла отправки

Использование образца отправки сохраняет требуемый порядок строк и схему. Замените только столбец `target` вероятностями положительного класса.

### TODO(student) — Задание 7: заполнить и сохранить файл отправки

Скопируйте `sample_submission` в `submission`, замените целевые значения на `test_probability`, проверьте схему из двух столбцов, число строк, точный порядок тестовых ID, уникальность ID, отсутствие пропусков и диапазон вероятностей, затем сохраните `submission.csv` с `index=False`.

In [14]:
submission = sample_submission.copy()
submission["target"] = test_probability

assert list(submission.columns) == ["row_id", "target"]
assert submission.shape == (1116, 2)
assert (submission["row_id"].to_numpy() == test["row_id"].to_numpy()).all()
assert (submission["row_id"].to_numpy() == sample_submission["row_id"].to_numpy()).all()
assert submission["row_id"].is_unique
assert not submission.isna().any().any()
assert submission["target"].between(0.0, 1.0).all()

submission.to_csv(DATA_DIR / "submission_baseline_tree.csv", index=False)

Предсказания базового дерева сохранены в `submission_baseline_tree.csv`, а не в `submission.csv`: файл `submission.csv` в конце ноутбука записывает только итоговая модель, выбранная в Задании 8 по валидационной выборке.

In [15]:
runtime_секунд = time.perf_counter() - notebook_started
print(f"ROC-AUC на валидации: {validation_auc:.6f}")
print(f"Время работы: {runtime_секунд:.2f} секунд")
print(submission.head())

ROC-AUC на валидации: 0.747309
Время работы: 0.69 секунд
                     row_id    target
0  row_46ad1afad8562239f79c  0.865625
1  row_b42ade9cd57b8d919acb  0.865625
2  row_c951cd5e197b6b4e8b83  0.611111
3  row_a54b3423760d599416ce  0.322105
4  row_a87c87cc47690e2a303f  0.322105


## TODO(student) — Задание 8: самостоятельное сравнение моделей

Обучите и сравните дерево решений, логистическую регрессию, случайный лес и градиентный бустинг, используя предоставленное разбиение train/validation. Представьте ROC-AUC на валидации и время работы каждой модели в компактной таблице, затем обоснуйте итоговый выбор. Не обучайте и не настраивайте модели на валидационных данных.

### Протокол исследования

* Все модели и все обучаемые шаги предобработки (в том числе статистики для инженерии признаков) подгоняются **только на `train.csv`**.
* `validation.csv` используется **только** для подсчёта ROC-AUC и выбора конфигурации; все числа ниже получены на одном и том же предоставленном валидационном разбиении. Выборки не объединяются и не перемешиваются.
* Тестовая выборка используется один раз — итоговой моделью, для получения вероятностей.
* Во всех случайных компонентах используется `SEED = 20260916`; для оценки устойчивости дополнительно используются пять фиксированных значений `SEED + 0 … SEED + 4`.

План: 8.1 понимание данных → 8.2 инженерия признаков → 8.3 одно интерпретируемое дерево → 8.4 объединение деревьев (лес, Extra Trees) → 8.5 градиентный бустинг → 8.6 референсные модели (логистическая регрессия, L1-сумма, SVM) → 8.7 стекинг: оценка SVM как признак для деревьев → 8.8 устойчивость к seed и усреднение моделей → 8.9 сводная таблица → 8.10 выбор итоговой модели → 8.11 итоговое обучение и файл отправки → 8.12 воспроизводимость → 10 как загрузить результат на Kaggle.

In [16]:
import sys
import warnings

import lightgbm as lgb
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import export_text

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.precision", 4)
pd.set_option("display.max_colwidth", None)
study_started = time.perf_counter()

### 8.1. Понимание данных

Прежде чем усложнять модель, полезно понять, что именно закодировано в 512 столбцах. Каждая строка описывает **пару** скрытых наблюдений, но самих наблюдений в таблице нет. Проверим три гипотезы: (1) признаки — это неотрицательные «расстояния» между двумя наблюдениями по отдельным координатам; (2) признаки не разбиты на две половины «наблюдение A / наблюдение B»; (3) простая сумма расстояний уже даёт заметный сигнал без всякого обучения.

In [17]:
print(f"Доля положительного класса: train {y_train.mean():.3f}, validation {y_validation.mean():.3f}")
print(
    "Минимальное значение признаков: "
    f"train {X_train.min():.4f}, validation {X_validation.min():.4f}, test {X_test.min():.4f}"
)
print(f"Среднее по признакам: {X_train.mean():.3f}, медиана: {np.median(X_train):.3f}, максимум: {X_train.max():.3f}")

univariate_auc = np.array([roc_auc_score(y_train, X_train[:, j]) for j in range(X_train.shape[1])])
print(
    "Одномерный ROC-AUC каждого признака на train: "
    f"min {univariate_auc.min():.3f}, медиана {np.median(univariate_auc):.3f}, max {univariate_auc.max():.3f}; "
    f"признаков с AUC < 0.5: {(univariate_auc < 0.5).sum()} из 512"
)

half = X_train.shape[1] // 2
paired_corr = np.array([np.corrcoef(X_train[:, j], X_train[:, j + half])[0, 1] for j in range(half)])
rng = np.random.default_rng(SEED)
random_pairs = rng.choice(X_train.shape[1], size=(half, 2))
random_pairs = random_pairs[random_pairs[:, 0] != random_pairs[:, 1]]
random_corr = np.array([np.corrcoef(X_train[:, i], X_train[:, j])[0, 1] for i, j in random_pairs])
print(
    f"Средняя корреляция f_j и f_(j+256): {paired_corr.mean():.3f}; "
    f"средняя корреляция случайных пар признаков: {random_corr.mean():.3f}"
)

l1_train_auc = roc_auc_score(y_train, -X_train.sum(axis=1))
l1_validation_auc = roc_auc_score(y_validation, -X_validation.sum(axis=1))
print(f"ROC-AUC оценки «минус L1-сумма» (без обучения): train {l1_train_auc:.4f}, validation {l1_validation_auc:.4f}")

Доля положительного класса: train 0.500, validation 0.500
Минимальное значение признаков: train 0.0000, validation 0.0000, test 0.0000
Среднее по признакам: 0.524, медиана: 0.304, максимум: 10.035


Одномерный ROC-AUC каждого признака на train: min 0.300, медиана 0.361, max 0.419; признаков с AUC < 0.5: 512 из 512
Средняя корреляция f_j и f_(j+256): 0.134; средняя корреляция случайных пар признаков: 0.126
ROC-AUC оценки «минус L1-сумма» (без обучения): train 0.8362, validation 0.8334


**Что мы видим.** Все 512 признаков неотрицательны, и у **каждого** из них одномерный ROC-AUC меньше 0.5: чем больше значение, тем вероятнее, что пара относится к разным сущностям. Это поведение расстояния между двумя наблюдениями по одной координате, а не «сырых» координат. Корреляция признака `f_j` со «зеркальным» `f_(j+256)` не выше, чем у случайной пары признаков, поэтому половин «A / B» здесь нет, и привычные парные признаки (разности, произведения, косинусное сходство двух половин) неприменимы. Классы сбалансированы (50 % / 50 %) и на train, и на validation.

Простейшая оценка без обучения — сумма всех расстояний со знаком минус — уже даёт ROC-AUC около 0.83 на валидации, то есть заметно выше базового дерева глубины 3 (0.747). Значит, полезно дать деревьям **агрегированные признаки строки**, которые они сами вычислить не могут (дерево делит пространство по одной координате за раз и не умеет суммировать 512 столбцов).

### 8.2. Инженерия признаков (статистики подгоняются только на train)

К 512 исходным столбцам добавляются 30 агрегатов строки:

* **16 агрегатов без обучения** — описание «профиля расстояний» пары: L1- и L2-норма, стандартное отклонение, максимум, квартили и 90-й перцентиль, число координат меньше 0.1 / 0.5 и больше 1 / 2, сумма логарифмов, сумма квадратных корней, среднее 16 наибольших и 16 наименьших значений.
* **14 агрегатов по группам информативности** — координаты ранжируются по одномерному ROC-AUC **на обучающей выборке**; для `k ∈ {32, 64, 128, 256}` считаются сумма по `k` самым информативным координатам, сумма по `k` наименее информативным и их отношение, а также взвешенные суммы с весами `(0.5 − AUC_j)^p`, `p ∈ {2, 4}`. Так неглубокое дерево получает доступ к «взвешенному расстоянию», которое иначе потребовало бы сотен разбиений.

Порядок координат и веса вычисляются один раз на `X_train` и затем применяются к валидационной и тестовой матрицам без пересчёта.

In [18]:
def fit_feature_stats(X, y):
    aucs = np.array([roc_auc_score(y, X[:, j]) for j in range(X.shape[1])])
    return {"order": np.argsort(aucs), "weights": np.clip(0.5 - aucs, 0.0, None)}


def make_features(X, stats, group_sizes=(32, 64, 128, 256), weight_powers=(2, 4)):
    sorted_rows = np.sort(X, axis=1)
    features = {name: X[:, j] for j, name in enumerate(feature_columns)}
    features.update(
        {
            "agg_l1": X.sum(axis=1),
            "agg_l2": np.sqrt((X**2).sum(axis=1)),
            "agg_std": X.std(axis=1),
            "agg_max": X.max(axis=1),
            "agg_q25": np.quantile(X, 0.25, axis=1),
            "agg_median": np.median(X, axis=1),
            "agg_q75": np.quantile(X, 0.75, axis=1),
            "agg_q90": np.quantile(X, 0.90, axis=1),
            "agg_n_below_0.1": (X < 0.1).sum(axis=1),
            "agg_n_below_0.5": (X < 0.5).sum(axis=1),
            "agg_n_above_1": (X > 1.0).sum(axis=1),
            "agg_n_above_2": (X > 2.0).sum(axis=1),
            "agg_log_sum": np.log(X + 1e-3).sum(axis=1),
            "agg_sqrt_sum": np.sqrt(X).sum(axis=1),
            "agg_top16_mean": sorted_rows[:, -16:].mean(axis=1),
            "agg_bottom16_mean": sorted_rows[:, :16].mean(axis=1),
        }
    )
    for k in group_sizes:
        top = X[:, stats["order"][:k]].sum(axis=1)
        bottom = X[:, stats["order"][-k:]].sum(axis=1)
        features[f"grp_top{k}_sum"] = top
        features[f"grp_bottom{k}_sum"] = bottom
        features[f"grp_top{k}_ratio"] = top / (bottom + 1e-6)
    for p in weight_powers:
        features[f"grp_weighted_sum_p{p}"] = (X * stats["weights"] ** p).sum(axis=1)
    return pd.DataFrame(features)


feature_stats = fit_feature_stats(X_train, y_train)
frame_train = make_features(X_train, feature_stats)
frame_validation = make_features(X_validation, feature_stats)
frame_test = make_features(X_test, feature_stats)

engineered_columns = list(frame_train.columns)
extra_columns = [column for column in engineered_columns if column not in feature_columns]
X_train_fe = frame_train.to_numpy(dtype=np.float64)
X_validation_fe = frame_validation.to_numpy(dtype=np.float64)
X_test_fe = frame_test.to_numpy(dtype=np.float64)

assert X_train_fe.shape == (2520, 542) and X_validation_fe.shape == (560, 542) and X_test_fe.shape == (1116, 542)
assert not np.isnan(X_train_fe).any() and not np.isnan(X_validation_fe).any() and not np.isnan(X_test_fe).any()

RAW = "512 исходных"
ENGINEERED = f"512 исходных + {len(extra_columns)} агрегатов"
FEATURE_SETS = {
    RAW: (X_train, X_validation, X_test, feature_columns),
    ENGINEERED: (X_train_fe, X_validation_fe, X_test_fe, engineered_columns),
}
print(f"Признаков после инженерии: {X_train_fe.shape[1]} ({ENGINEERED})")

aggregate_quality = pd.DataFrame(
    {
        "Агрегат": extra_columns,
        "ROC-AUC (train)": [roc_auc_score(y_train, frame_train[c]) for c in extra_columns],
        "ROC-AUC (валидация)": [roc_auc_score(y_validation, frame_validation[c]) for c in extra_columns],
    }
)
for column in ("ROC-AUC (train)", "ROC-AUC (валидация)"):
    aggregate_quality[column] = np.maximum(aggregate_quality[column], 1 - aggregate_quality[column])
display(aggregate_quality.sort_values("ROC-AUC (train)", ascending=False).reset_index(drop=True).round(4))

Признаков после инженерии: 542 (512 исходных + 30 агрегатов)


,Агрегат,ROC-AUC (train),ROC-AUC (валидация)
0,grp_top64_sum,0.8778,0.8702
1,grp_top128_sum,0.8776,0.8695
2,grp_top32_sum,0.8745,0.8669
3,grp_top256_sum,0.8661,0.8583
4,grp_weighted_sum_p4,0.8609,0.8547
5,grp_weighted_sum_p2,0.8499,0.8447
6,agg_n_below_0.5,0.8371,0.8319
7,agg_sqrt_sum,0.8366,0.8316
8,agg_l1,0.8362,0.8334
9,agg_n_above_1,0.8356,0.8342


Каждый агрегат в отдельности — это одномерная оценка (AUC приведён к виду «больше 0.5», направление у всех очевидно: больше расстояние — меньше вероятность совпадения). Лучшие агрегаты по группам информативности (`grp_top64_sum`, взвешенные суммы) дают около 0.87 на валидации — больше, чем простая L1-сумма (0.83), потому что они не учитывают «шумовые» координаты, слабо связанные с идентичностью.

### Единый способ учёта экспериментов

Каждая конфигурация обучается на `train`, для неё замеряются время обучения и время предсказания на валидации, ROC-AUC на train и на validation. Результаты складываются в общий список для сводной таблицы (раздел 8.9); столбец «Статус» отличает кандидатов на итоговую модель (деревья и их ансамбли) от референсных моделей вне класса моделей задания. Валидационные оценки сохраняются, чтобы позже посчитать усреднение моделей без повторного обучения.

In [19]:
experiments = []
validation_predictions = {}
fitted_models = {}


CANDIDATE = "кандидат"
REFERENCE = "референс, вне класса моделей задания"


def model_scores(estimator, X):
    if hasattr(estimator, "predict_proba"):
        return estimator.predict_proba(X)[:, 1]
    return estimator.decision_function(X)


def run_experiment(name, features, make_model, seed=SEED, record=True, candidate=True):
    X_tr, X_va, _, _ = FEATURE_SETS[features]
    estimator = make_model(seed)
    started = time.perf_counter()
    estimator.fit(X_tr, y_train)
    fit_seconds = time.perf_counter() - started
    started = time.perf_counter()
    proba_validation = model_scores(estimator, X_va)
    predict_seconds = time.perf_counter() - started
    proba_train = model_scores(estimator, X_tr)
    row = {
        "Модель": name,
        "Признаки": features,
        "Статус": CANDIDATE if candidate else REFERENCE,
        "ROC-AUC (train)": roc_auc_score(y_train, proba_train),
        "ROC-AUC (валидация)": roc_auc_score(y_validation, proba_validation),
        "Время обучения, с": fit_seconds,
        "Время предсказания, с": predict_seconds,
        "seed": seed,
    }
    if record:
        experiments.append(row)
        validation_predictions[(name, features)] = proba_validation
        fitted_models[(name, features)] = estimator
        print(
            f"{row['ROC-AUC (валидация)']:.4f} (train {row['ROC-AUC (train)']:.4f}, "
            f"{fit_seconds:6.1f} с)  {name} — {features}"
        )
    return row, estimator, proba_validation

### 8.3. Сначала одно интерпретируемое дерево

Одно дерево — самая прозрачная модель: его правила можно прочитать. Посмотрим, как глубина (`max_depth`) и минимальный размер листа (`min_samples_leaf`) влияют на качество на train и на validation, и как растёт разрыв между ними. Для каждого дерева фиксируем фактическую глубину, число листьев, минимальный и медианный размер листа, оба ROC-AUC и время обучения. Сетка обучается на обоих наборах признаков.

In [20]:
def leaf_sizes(tree):
    structure = tree.tree_
    return structure.n_node_samples[structure.children_left == -1]


tree_rows = []
for features in FEATURE_SETS:
    X_tr, X_va, _, _ = FEATURE_SETS[features]
    for max_depth in (1, 2, 3, 4, 5, 6, 8, 10, None):
        for min_samples_leaf in (1, 5, 20, 50):
            started = time.perf_counter()
            tree = DecisionTreeClassifier(
                max_depth=max_depth, min_samples_leaf=min_samples_leaf, random_state=SEED
            ).fit(X_tr, y_train)
            fit_seconds = time.perf_counter() - started
            sizes = leaf_sizes(tree)
            train_auc = roc_auc_score(y_train, tree.predict_proba(X_tr)[:, 1])
            validation_auc_tree = roc_auc_score(y_validation, tree.predict_proba(X_va)[:, 1])
            tree_rows.append(
                {
                    "Признаки": features,
                    "max_depth": "None" if max_depth is None else max_depth,
                    "min_samples_leaf": min_samples_leaf,
                    "Глубина": tree.get_depth(),
                    "Листьев": tree.get_n_leaves(),
                    "Мин. лист": int(sizes.min()),
                    "Медианный лист": float(np.median(sizes)),
                    "ROC-AUC (train)": train_auc,
                    "ROC-AUC (валидация)": validation_auc_tree,
                    "Разрыв train − validation": train_auc - validation_auc_tree,
                    "Время обучения, с": fit_seconds,
                }
            )
tree_table = pd.DataFrame(tree_rows)
print(f"Обучено деревьев: {len(tree_table)}")
display(tree_table.round(4))

Обучено деревьев: 72


,Признаки,max_depth,min_samples_leaf,Глубина,Листьев,Мин. лист,Медианный лист,ROC-AUC (train),ROC-AUC (валидация),Разрыв train − validation,"Время обучения, с"
0,512 исходных,1,1,1,2,1162,1260.0,0.6516,0.6571,-0.0056,0.1063
1,512 исходных,1,5,1,2,1162,1260.0,0.6516,0.6571,-0.0056,0.1040
2,512 исходных,1,20,1,2,1162,1260.0,0.6516,0.6571,-0.0056,0.1046
3,512 исходных,1,50,1,2,1162,1260.0,0.6516,0.6571,-0.0056,0.1038
4,512 исходных,2,1,2,4,417,679.0,0.7270,0.7196,0.0074,0.1961
5,512 исходных,2,5,2,4,417,679.0,0.7270,0.7196,0.0074,0.1981
6,512 исходных,2,20,2,4,417,679.0,0.7270,0.7196,0.0074,0.1985
7,512 исходных,2,50,2,4,417,679.0,0.7270,0.7196,0.0074,0.1942
8,512 исходных,3,1,3,8,87,271.5,0.7904,0.7473,0.0431,0.2884
9,512 исходных,3,5,3,8,87,271.5,0.7904,0.7473,0.0431,0.2887


In [21]:
depth_order = ["1", "2", "3", "4", "5", "6", "8", "10", "None"]
tree_table["max_depth"] = pd.Categorical(tree_table["max_depth"].astype(str), categories=depth_order, ordered=True)
for value in ("ROC-AUC (валидация)", "Разрыв train − validation"):
    print(f"\n{value}: строки — max_depth, столбцы — (признаки, min_samples_leaf)")
    display(
        tree_table.pivot_table(index="max_depth", columns=["Признаки", "min_samples_leaf"], values=value, observed=False).round(4)
    )


ROC-AUC (валидация): строки — max_depth, столбцы — (признаки, min_samples_leaf)


Признаки         512 исходных                         512 исходных + 30 агрегатов                        
min_samples_leaf           1       5       20      50                          1       5       20      50
max_depth                                                                                                
1                      0.6571  0.6571  0.6571  0.6571                      0.7964  0.7964  0.7964  0.7964
2                      0.7196  0.7196  0.7196  0.7196                      0.8606  0.8606  0.8606  0.8606
3                      0.7473  0.7473  0.7473  0.7473                      0.8649  0.8649  0.8645  0.8693
4                      0.7676  0.7610  0.7659  0.7673                      0.8584  0.8599  0.8589  0.8502
5                      0.7836  0.7774  0.7879  0.7853                      0.8596  0.8580  0.8661  0.8601
6                      0.7825  0.7763  0.7856  0.7942                      0.8363  0.8461  0.8513  0.8571
8                      0.7210  0.7297  0.7880  0.7947                      0.8085  0.8125  0.8445  0.8623
10                     0.6505  0.6886  0.7848  0.7947                      0.7437  0.7828  0.8395  0.8623
None                   0.6946  0.7081  0.7848  0.7947                      0.7339  0.8011  0.8392  0.8623


Разрыв train − validation: строки — max_depth, столбцы — (признаки, min_samples_leaf)


Признаки         512 исходных                         512 исходных + 30 агрегатов                        
min_samples_leaf           1       5       20      50                          1       5       20      50
max_depth                                                                                                
1                     -0.0056 -0.0056 -0.0056 -0.0056                      0.0000  0.0000  0.0000  0.0000
2                      0.0074  0.0074  0.0074  0.0074                      0.0038  0.0038  0.0038  0.0038
3                      0.0431  0.0431  0.0431  0.0431                      0.0260  0.0260  0.0264  0.0234
4                      0.0785  0.0845  0.0793  0.0728                      0.0543  0.0537  0.0508  0.0635
5                      0.1095  0.1146  0.1026  0.0808                      0.0746  0.0767  0.0602  0.0652
6                      0.1499  0.1555  0.1334  0.0860                      0.1161  0.1050  0.0899  0.0751
8                      0.2630  0.2479  0.1514  0.0881                      0.1680  0.1620  0.1135  0.0734
10                     0.3465  0.3006  0.1560  0.0881                      0.2452  0.2050  0.1230  0.0734
None                   0.3054  0.2839  0.1560  0.0881                      0.2661  0.1929  0.1235  0.0734

In [22]:
best_trees = (
    tree_table.sort_values("ROC-AUC (валидация)", ascending=False)
    .groupby("Признаки", observed=True)
    .head(1)
    .reset_index(drop=True)
)
print("Лучшее дерево на каждом наборе признаков (по валидации):")
display(best_trees.round(4))

for features in FEATURE_SETS:
    run_experiment(
        "Дерево решений (max_depth=3, min_samples_leaf=20) — базовая модель",
        features,
        lambda seed: DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=seed),
    )
    row = best_trees[best_trees["Признаки"] == features].iloc[0]
    best_depth = None if row["max_depth"] == "None" else int(row["max_depth"])
    best_leaf = int(row["min_samples_leaf"])
    run_experiment(
        f"Дерево решений (max_depth={row['max_depth']}, min_samples_leaf={best_leaf}) — лучшее в сетке",
        features,
        lambda seed, d=best_depth, l=best_leaf: DecisionTreeClassifier(max_depth=d, min_samples_leaf=l, random_state=seed),
    )
    run_experiment(
        "Дерево решений без ограничений (max_depth=None, min_samples_leaf=1)",
        features,
        lambda seed: DecisionTreeClassifier(random_state=seed),
    )

Лучшее дерево на каждом наборе признаков (по валидации):


,Признаки,max_depth,min_samples_leaf,Глубина,Листьев,Мин. лист,Медианный лист,ROC-AUC (train),ROC-AUC (валидация),Разрыв train − validation,"Время обучения, с"
0,512 исходных + 30 агрегатов,3,50,3,8,62,233.5,0.8927,0.8693,0.0234,0.2837
1,512 исходных,8,50,8,33,50,67.0,0.8829,0.7947,0.0881,0.4603


0.7473 (train 0.7904,    0.3 с)  Дерево решений (max_depth=3, min_samples_leaf=20) — базовая модель — 512 исходных


0.7947 (train 0.8829,    0.5 с)  Дерево решений (max_depth=8, min_samples_leaf=50) — лучшее в сетке — 512 исходных


0.6946 (train 1.0000,    0.9 с)  Дерево решений без ограничений (max_depth=None, min_samples_leaf=1) — 512 исходных


0.8645 (train 0.8909,    0.3 с)  Дерево решений (max_depth=3, min_samples_leaf=20) — базовая модель — 512 исходных + 30 агрегатов


0.8693 (train 0.8927,    0.3 с)  Дерево решений (max_depth=3, min_samples_leaf=50) — лучшее в сетке — 512 исходных + 30 агрегатов


0.7339 (train 1.0000,    1.4 с)  Дерево решений без ограничений (max_depth=None, min_samples_leaf=1) — 512 исходных + 30 агрегатов


#### Правила дерева глубины 3

Базовое дерево (`model` из Задания 4) на исходных признаках и такое же дерево на расширенном наборе. Веса в листьях — число объектов классов `[0, 1]`.

In [23]:
print("Дерево глубины 3 на исходных признаках (model из Задания 4):")
print(export_text(model, feature_names=feature_columns, decimals=3, show_weights=True))
importances_raw = pd.Series(model.feature_importances_, index=feature_columns)
print("Важность признаков (топ-5):")
print(importances_raw.sort_values(ascending=False).head(5).round(3).to_string())
print(
    "Одномерный ROC-AUC этих признаков на train:",
    {name: round(univariate_auc[feature_columns.index(name)], 3) for name in importances_raw.sort_values(ascending=False).head(3).index},
)

Дерево глубины 3 на исходных признаках (model из Задания 4):
|--- f0289 <= 0.493
|   |--- f0443 <= 0.537
|   |   |--- f0489 <= 0.820
|   |   |   |--- weights: [86.000, 554.000] class: 1
|   |   |--- f0489 >  0.820
|   |   |   |--- weights: [57.000, 30.000] class: 0
|   |--- f0443 >  0.537
|   |   |--- f0056 <= 0.374
|   |   |   |--- weights: [153.000, 209.000] class: 1
|   |   |--- f0056 >  0.374
|   |   |   |--- weights: [192.000, 77.000] class: 0
|--- f0289 >  0.493
|   |--- f0109 <= 0.666
|   |   |--- f0203 <= 0.543
|   |   |   |--- weights: [105.000, 165.000] class: 1
|   |   |--- f0203 >  0.543
|   |   |   |--- weights: [322.000, 153.000] class: 0
|   |--- f0109 >  0.666
|   |   |--- f0304 <= 0.314
|   |   |   |--- weights: [95.000, 49.000] class: 0
|   |   |--- f0304 >  0.314
|   |   |   |--- weights: [250.000, 23.000] class: 0

Важность признаков (топ-5):
f0289    0.340
f0443    0.242
f0489    0.121
f0109    0.101
f0203    0.084
Одномерный ROC-AUC этих признаков на train: {'f028

In [24]:
tree_fe = DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=SEED).fit(X_train_fe, y_train)
print("Дерево глубины 3 на расширенном наборе признаков:")
print(export_text(tree_fe, feature_names=engineered_columns, decimals=3, show_weights=True))
importances_fe = pd.Series(tree_fe.feature_importances_, index=engineered_columns)
print("Важность признаков (топ-5):")
print(importances_fe.sort_values(ascending=False).head(5).round(3).to_string())
print(f"ROC-AUC на валидации: {roc_auc_score(y_validation, tree_fe.predict_proba(X_validation_fe)[:, 1]):.4f}")

Дерево глубины 3 на расширенном наборе признаков:
|--- grp_top64_sum <= 40.528
|   |--- grp_top64_sum <= 36.269
|   |   |--- grp_top256_ratio <= 1.678
|   |   |   |--- weights: [32.000, 732.000] class: 1
|   |   |--- grp_top256_ratio >  1.678
|   |   |   |--- weights: [9.000, 13.000] class: 1
|   |--- grp_top64_sum >  36.269
|   |   |--- grp_bottom128_sum <= 41.191
|   |   |   |--- weights: [24.000, 6.000] class: 0
|   |   |--- grp_bottom128_sum >  41.191
|   |   |   |--- weights: [40.000, 101.000] class: 1
|--- grp_top64_sum >  40.528
|   |--- grp_top128_ratio <= 1.502
|   |   |--- agg_log_sum <= -479.100
|   |   |   |--- weights: [298.000, 273.000] class: 0
|   |   |--- agg_log_sum >  -479.100
|   |   |   |--- weights: [162.000, 39.000] class: 0
|   |--- grp_top128_ratio >  1.502
|   |   |--- agg_log_sum <= -704.908
|   |   |   |--- weights: [71.000, 43.000] class: 0
|   |   |--- agg_log_sum >  -704.908
|   |   |   |--- weights: [624.000, 53.000] class: 0

Важность признаков (топ-5):

**Что показывает сетка деревьев.**

* На исходных признаках дерево глубины 1 даёт 0.657, глубины 3 — 0.747 (базовая модель), лучший результат в сетке — 0.795 при `max_depth=8, min_samples_leaf=50`. Дальше рост глубины только вредит: дерево без ограничений (глубина 22, 201 лист, медианный лист — 3 объекта) идеально запоминает train (ROC-AUC 1.000), но на валидации падает до 0.695 — разрыв 0.305.
* Разрыв train − validation монотонно растёт с глубиной: 0.04 при глубине 3, 0.08 при 4, 0.11 при 5, 0.15 при 6, 0.26 при 8 и 0.35 при 10 (для `min_samples_leaf=1`). Ограничение размера листа сдерживает переобучение: при `min_samples_leaf=50` разрыв не превышает 0.09 даже без ограничения глубины, потому что дерево физически не может вырасти глубже 8 уровней и 33 листьев.
* Агрегаты меняют картину сильнее любого гиперпараметра: уже «пень» (глубина 1) по `grp_top64_sum` даёт 0.796 — больше, чем лучшее дерево любой глубины на исходных признаках. Дерево глубины 3 — 0.865–0.869, и это оптимум: с глубины 4 валидационное качество начинает снижаться, а разрыв — расти (0.27 у дерева без ограничений при 0.734 на валидации).
* Выбор «лучшего дерева в сетке» сделан по валидации, поэтому его оценка слегка оптимистична — это относится ко всем моделям, выбранным из нескольких конфигураций.
* Время обучения одного дерева — доли секунды на любом наборе признаков.

**На что смотрит дерево.** На исходных признаках корень делит по `f0289` (одномерный AUC 0.308 — одна из самых информативных координат), затем по `f0443`, `f0489`, `f0109`, `f0203`; во всех узлах правило одно и то же — «маленькое расстояние по координате → класс 1, большое → класс 0». Но дерево видит лишь 7 из 512 координат, поэтому пары, различающиеся по другим координатам, оно ранжирует почти наугад. На расширенном наборе 80 % важности приходится на `grp_top64_sum` — сумму расстояний по 64 самым информативным координатам: первое же правило `grp_top64_sum ≤ 40.5` отделяет 957 обучающих объектов, среди которых 89 % положительных, а лист `grp_top64_sum ≤ 36.3` при небольшом `grp_top256_ratio` содержит 764 объекта с 96 % положительных. Остальные разбиения используют отношение `grp_top128_ratio` (какая доля расстояния приходится на информативные координаты) и `agg_log_sum`. Такое дерево читается как одно правило «суммарное расстояние по ключевым координатам мало → одна и та же сущность» и даёт 0.865 против 0.747 у базовой модели.

### 8.4. Что меняется при объединении деревьев

Полностью выращенное дерево запоминает обучающую выборку (ROC-AUC на train = 1.0), но его предсказания сильно зависят от конкретной выборки — у него большая **дисперсия**. Четвёртая часть этого раздела — устойчивость к seed — вынесена в 8.8, потому что в ней участвуют и модели бустинга из 8.5 и стекинга из 8.7. Случайный лес обучает много таких деревьев на бутстрэп-подвыборках со случайным подмножеством признаков в каждом узле и **усредняет** их вероятности. Усреднение `n` слабо коррелированных оценок уменьшает дисперсию примерно в `n` раз (для полностью независимых оценок — ровно в `n` раз), а смещение каждого дерева остаётся прежним. Ниже это показано в числах.

#### (i) ROC-AUC в зависимости от числа деревьев

Лес из 800 деревьев обучается один раз, а затем предсказания первых `n` деревьев усредняются — это в точности то, что вернул бы `RandomForestClassifier(n_estimators=n)` с тем же `random_state`.

In [25]:
forest_sizes = [1, 5, 10, 25, 50, 100, 200, 400, 800]
forest_rows = []
single_tree_summary = []
for features in FEATURE_SETS:
    X_tr, X_va, _, _ = FEATURE_SETS[features]
    _, forest, _ = run_experiment(
        "Случайный лес (800 деревьев, max_features=sqrt)",
        features,
        lambda seed: RandomForestClassifier(n_estimators=800, random_state=seed, n_jobs=-1),
    )
    per_tree_validation = np.array([tree.predict_proba(X_va)[:, 1] for tree in forest.estimators_])
    per_tree_train = np.array([tree.predict_proba(X_tr)[:, 1] for tree in forest.estimators_])
    single_aucs = np.array([roc_auc_score(y_validation, p) for p in per_tree_validation])
    single_tree_summary.append(
        {
            "Признаки": features,
            "Одно дерево леса: среднее ROC-AUC (валидация)": single_aucs.mean(),
            "Одно дерево леса: std ROC-AUC": single_aucs.std(),
            "Лес из 800 деревьев: ROC-AUC (валидация)": roc_auc_score(y_validation, per_tree_validation.mean(axis=0)),
        }
    )
    for n in forest_sizes:
        forest_rows.append(
            {
                "Признаки": features,
                "Деревьев": n,
                "ROC-AUC (train)": roc_auc_score(y_train, per_tree_train[:n].mean(axis=0)),
                "ROC-AUC (валидация)": roc_auc_score(y_validation, per_tree_validation[:n].mean(axis=0)),
            }
        )
forest_curve = pd.DataFrame(forest_rows).pivot(index="Деревьев", columns="Признаки", values=["ROC-AUC (train)", "ROC-AUC (валидация)"])
display(forest_curve.round(4))
display(pd.DataFrame(single_tree_summary).round(4))

0.9009 (train 1.0000,    5.0 с)  Случайный лес (800 деревьев, max_features=sqrt) — 512 исходных


0.9178 (train 1.0000,    5.4 с)  Случайный лес (800 деревьев, max_features=sqrt) — 512 исходных + 30 агрегатов


ROC-AUC (train)                             ROC-AUC (валидация)                            
Признаки    512 исходных 512 исходных + 30 агрегатов        512 исходных 512 исходных + 30 агрегатов
Деревьев                                                                                            
1                 0.8770                      0.8948              0.6875                      0.7536
5                 0.9959                      0.9957              0.7941                      0.8656
10                0.9998                      0.9997              0.8304                      0.8818
25                1.0000                      1.0000              0.8666                      0.9026
50                1.0000                      1.0000              0.8921                      0.9071
100               1.0000                      1.0000              0.8981                      0.9136
200               1.0000                      1.0000              0.9032                      0.9191
400               1.0000                      1.0000              0.9024                      0.9179
800               1.0000                      1.0000              0.9009                      0.9178

,Признаки,Одно дерево леса: среднее ROC-AUC (валидация),Одно дерево леса: std ROC-AUC,Лес из 800 деревьев: ROC-AUC (валидация)
0,512 исходных,0.6584,0.0201,0.9009
1,512 исходных + 30 агрегатов,0.7275,0.0192,0.9178


#### (ii) Число признаков в узле (`max_features`) и (iii) Extra Trees против случайного леса

`max_features` управляет тем, насколько деревья леса различаются между собой: чем меньше признаков рассматривается в узле, тем слабее коррелированы деревья (сильнее эффект усреднения), но тем хуже каждое отдельное дерево. Extra Trees дополнительно выбирают порог разбиения случайно, а не оптимально, — деревья ещё разнообразнее и обучаются заметно быстрее. Здесь используется 400 деревьев: по кривой выше видно, что после 200–400 деревьев качество леса уже не меняется.

In [26]:
for max_features in ("sqrt", 0.1, 0.2):
    run_experiment(
        f"Случайный лес (400 деревьев, max_features={max_features})",
        ENGINEERED,
        lambda seed, mf=max_features: RandomForestClassifier(n_estimators=400, max_features=mf, random_state=seed, n_jobs=-1),
    )
for max_features in ("sqrt", 0.1, 0.2):
    run_experiment(
        f"Extra Trees (400 деревьев, max_features={max_features})",
        ENGINEERED,
        lambda seed, mf=max_features: ExtraTreesClassifier(n_estimators=400, max_features=mf, random_state=seed, n_jobs=-1),
    )
run_experiment(
    "Extra Trees (400 деревьев, max_features=0.2)",
    RAW,
    lambda seed: ExtraTreesClassifier(n_estimators=400, max_features=0.2, random_state=seed, n_jobs=-1),
)
_ = run_experiment(
    "Случайный лес (200 деревьев, max_features=sqrt) — из первой попытки Задания 8",
    RAW,
    lambda seed: RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1),
)

0.9179 (train 1.0000,    2.6 с)  Случайный лес (400 деревьев, max_features=sqrt) — 512 исходных + 30 агрегатов


0.9204 (train 1.0000,    6.4 с)  Случайный лес (400 деревьев, max_features=0.1) — 512 исходных + 30 агрегатов


0.9220 (train 1.0000,   13.1 с)  Случайный лес (400 деревьев, max_features=0.2) — 512 исходных + 30 агрегатов


0.9219 (train 1.0000,    0.8 с)  Extra Trees (400 деревьев, max_features=sqrt) — 512 исходных + 30 агрегатов


0.9255 (train 1.0000,    1.1 с)  Extra Trees (400 деревьев, max_features=0.1) — 512 исходных + 30 агрегатов


0.9281 (train 1.0000,    1.8 с)  Extra Trees (400 деревьев, max_features=0.2) — 512 исходных + 30 агрегатов


0.9164 (train 1.0000,    2.2 с)  Extra Trees (400 деревьев, max_features=0.2) — 512 исходных


0.9032 (train 1.0000,    1.5 с)  Случайный лес (200 деревьев, max_features=sqrt) — из первой попытки Задания 8 — 512 исходных


**Почему усреднение помогает и как это видно в числах.**

* Отдельное дерево леса (полностью выращенное, на бутстрэп-подвыборке, со случайным подмножеством признаков в узлах) — слабый и нестабильный классификатор: в среднем 0.658 ± 0.020 на исходных признаках и 0.728 ± 0.019 на расширенных. Это хуже дерева глубины 3, потому что каждое такое дерево переобучено под свою подвыборку.
* Ошибки разных деревьев мало коррелированы, поэтому их среднее гораздо точнее: 5 деревьев — 0.794 / 0.866, 25 — 0.867 / 0.903, 100 — 0.898 / 0.914, 200 — 0.903 / 0.919 (исходные / расширенные признаки). После 200–400 деревьев кривая выходит на плато (400 и 800 деревьев дают те же 0.902 / 0.918): дисперсия уже подавлена, а смещение усреднением не убирается. ROC-AUC на train равен 1.000 начиная с 25 деревьев — лес запоминает обучающую выборку, но, в отличие от одного дерева, не теряет качество на валидации, потому что переобучение отдельных деревьев усредняется, а не накапливается.
* `max_features`: чем больше признаков рассматривается в узле, тем сильнее каждое дерево и тем дороже обучение — 0.918 (sqrt ≈ 23 признака, 2.7 с) → 0.920 (10 %, 6.6 с) → 0.922 (20 %, 13.1 с). Выигрыш 0.004 стоит пятикратного времени и лежит в пределах шума оценки.
* Extra Trees при тех же параметрах и точнее, и в 3–7 раз быстрее: 0.922 / 0.926 / 0.928 против 0.918 / 0.920 / 0.922 у леса при 0.8 / 1.1 / 1.8 с обучения. Случайные пороги делают деревья ещё менее коррелированными (эффект усреднения сильнее) и избавляют от перебора порогов при обучении. На исходных признаках Extra Trees дают 0.916: агрегаты добавляют около 0.012 и ансамблю.
* Случайный лес из первой попытки Задания 8 (200 деревьев, исходные признаки) воспроизводится точно — 0.9032.

### 8.5. Градиентный бустинг

Бустинг строит деревья **последовательно**: каждое следующее исправляет ошибки суммы предыдущих, поэтому деревья здесь неглубокие, а число итераций и шаг обучения (`learning_rate`) — главные регуляторы недо-/переобучения. Классический `GradientBoostingClassifier` из scikit-learn перебирает все пороги всех признаков и работает в один поток: в первой попытке Задания 8 он обучался около 74 с (0.9051 на валидации). Гистограммные реализации — `HistGradientBoostingClassifier` (scikit-learn) и LightGBM — квантуют признаки в 255 бинов и распараллеливаются, поэтому на порядок быстрее при том же или лучшем качестве.

LightGBM запускается с `deterministic=True` и `force_row_wise=True`; число потоков фиксировано (`n_jobs=4`), потому что результат LightGBM гарантированно повторяется только при одинаковом числе потоков.

In [27]:
def make_hgb(learning_rate, max_iter, max_leaf_nodes):
    return lambda seed: HistGradientBoostingClassifier(
        learning_rate=learning_rate, max_iter=max_iter, max_leaf_nodes=max_leaf_nodes, random_state=seed
    )


def make_lgbm(learning_rate, n_estimators, num_leaves, colsample_bytree):
    return lambda seed: lgb.LGBMClassifier(
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        num_leaves=num_leaves,
        colsample_bytree=colsample_bytree,
        subsample=0.8,
        subsample_freq=1,
        min_child_samples=20,
        random_state=seed,
        deterministic=True,
        force_row_wise=True,
        n_jobs=4,
        verbose=-1,
    )


run_experiment(
    "GradientBoostingClassifier (параметры по умолчанию) — из первой попытки Задания 8",
    RAW,
    lambda seed: GradientBoostingClassifier(random_state=seed),
)

HGB_BEST = "HistGradientBoosting (lr=0.05, 400 итераций, 31 лист)"
LGBM_BEST = "LightGBM (lr=0.05, 400 итераций, 31 лист, colsample=0.5)"

run_experiment(HGB_BEST, RAW, make_hgb(0.05, 400, 31))
run_experiment("HistGradientBoosting (lr=0.05, 300 итераций, 7 листьев)", ENGINEERED, make_hgb(0.05, 300, 7))
run_experiment("HistGradientBoosting (lr=0.03, 800 итераций, 15 листьев)", ENGINEERED, make_hgb(0.03, 800, 15))
run_experiment(HGB_BEST, ENGINEERED, make_hgb(0.05, 400, 31))

run_experiment(LGBM_BEST, RAW, make_lgbm(0.05, 400, 31, 0.5))
run_experiment("LightGBM (lr=0.03, 600 итераций, 7 листьев, colsample=0.3)", ENGINEERED, make_lgbm(0.03, 600, 7, 0.3))
run_experiment("LightGBM (lr=0.03, 600 итераций, 15 листьев, colsample=0.5)", ENGINEERED, make_lgbm(0.03, 600, 15, 0.5))
run_experiment("LightGBM (lr=0.02, 1000 итераций, 15 листьев, colsample=0.3)", ENGINEERED, make_lgbm(0.02, 1000, 15, 0.3))
run_experiment("LightGBM (lr=0.03, 800 итераций, 15 листьев, colsample=0.2)", ENGINEERED, make_lgbm(0.03, 800, 15, 0.2))
_ = run_experiment(LGBM_BEST, ENGINEERED, make_lgbm(0.05, 400, 31, 0.5))

0.9051 (train 0.9973,   22.4 с)  GradientBoostingClassifier (параметры по умолчанию) — из первой попытки Задания 8 — 512 исходных


0.9202 (train 1.0000,    4.6 с)  HistGradientBoosting (lr=0.05, 400 итераций, 31 лист) — 512 исходных


0.9200 (train 0.9997,    1.1 с)  HistGradientBoosting (lr=0.05, 300 итераций, 7 листьев) — 512 исходных + 30 агрегатов


0.9262 (train 1.0000,    4.8 с)  HistGradientBoosting (lr=0.03, 800 итераций, 15 листьев) — 512 исходных + 30 агрегатов


0.9300 (train 1.0000,    4.7 с)  HistGradientBoosting (lr=0.05, 400 итераций, 31 лист) — 512 исходных + 30 агрегатов


...

0.9230 (train 0.9999,    0.8 с)  LightGBM (lr=0.03, 600 итераций, 7 листьев, colsample=0.3) — 512 исходных + 30 агрегатов


0.9291 (train 1.0000,    2.3 с)  LightGBM (lr=0.03, 600 итераций, 15 листьев, colsample=0.5) — 512 исходных + 30 агрегатов


0.9305 (train 1.0000,    2.4 с)  LightGBM (lr=0.02, 1000 итераций, 15 листьев, colsample=0.3) — 512 исходных + 30 агрегатов


0.9299 (train 1.0000,    1.4 с)  LightGBM (lr=0.03, 800 итераций, 15 листьев, colsample=0.2) — 512 исходных + 30 агрегатов


0.9314 (train 1.0000,    2.5 с)  LightGBM (lr=0.05, 400 итераций, 31 лист, colsample=0.5) — 512 исходных + 30 агрегатов


#### Качество в зависимости от числа итераций и шага обучения

`staged_predict_proba` возвращает предсказания после каждой итерации бустинга без повторного обучения, поэтому кривые «ROC-AUC от числа деревьев» получаются из одной модели. Сравниваются три шага обучения при одинаковых 400 итерациях и 31 листе: маленький шаг недообучается за 400 итераций, большой — быстро достигает максимума и начинает переобучаться. Для LightGBM аналогичная кривая строится через параметр `num_iteration` при предсказании.

In [28]:
checkpoints = [10, 25, 50, 100, 200, 300, 400]
curve_rows = []
for learning_rate in (0.02, 0.05, 0.2):
    booster = HistGradientBoostingClassifier(
        learning_rate=learning_rate, max_iter=400, max_leaf_nodes=31, random_state=SEED
    ).fit(X_train_fe, y_train)
    train_curve = np.array([roc_auc_score(y_train, p[:, 1]) for p in booster.staged_predict_proba(X_train_fe)])
    validation_curve = np.array(
        [roc_auc_score(y_validation, p[:, 1]) for p in booster.staged_predict_proba(X_validation_fe)]
    )
    best_iteration = int(validation_curve.argmax()) + 1
    for n in checkpoints:
        curve_rows.append(
            {
                "Модель": f"HistGradientBoosting, lr={learning_rate}",
                "Итераций": n,
                "ROC-AUC (train)": train_curve[n - 1],
                "ROC-AUC (валидация)": validation_curve[n - 1],
            }
        )
    print(
        f"HistGradientBoosting, lr={learning_rate}: максимум на валидации {validation_curve.max():.4f} "
        f"на итерации {best_iteration}; после 400 итераций {validation_curve[-1]:.4f} (train {train_curve[-1]:.4f})"
    )

lgbm_model = fitted_models[(LGBM_BEST, ENGINEERED)]
for n in checkpoints:
    curve_rows.append(
        {
            "Модель": "LightGBM, lr=0.05",
            "Итераций": n,
            "ROC-AUC (train)": roc_auc_score(y_train, lgbm_model.predict_proba(X_train_fe, num_iteration=n)[:, 1]),
            "ROC-AUC (валидация)": roc_auc_score(
                y_validation, lgbm_model.predict_proba(X_validation_fe, num_iteration=n)[:, 1]
            ),
        }
    )
boosting_curve = pd.DataFrame(curve_rows).pivot(index="Итераций", columns="Модель", values=["ROC-AUC (train)", "ROC-AUC (валидация)"])
display(boosting_curve.round(4))

HistGradientBoosting, lr=0.02: максимум на валидации 0.9285 на итерации 398; после 400 итераций 0.9284 (train 1.0000)


HistGradientBoosting, lr=0.05: максимум на валидации 0.9300 на итерации 400; после 400 итераций 0.9300 (train 1.0000)


HistGradientBoosting, lr=0.2: максимум на валидации 0.9289 на итерации 102; после 400 итераций 0.9282 (train 1.0000)


ROC-AUC (train)                                                                                        ROC-AUC (валидация)                                \
Модель   HistGradientBoosting, lr=0.02 HistGradientBoosting, lr=0.05 HistGradientBoosting, lr=0.2 LightGBM, lr=0.05 HistGradientBoosting, lr=0.02 HistGradientBoosting, lr=0.05   
Итераций                                                                                                                                                                          
10                              0.9615                        0.9748                       0.9948            0.9751                        0.8870                        0.9060   
25                              0.9753                        0.9885                       1.0000            0.9876                        0.9057                        0.9151   
50                              0.9855                        0.9975                       1.0000            0.9964                        0.9116                        0.9202   
100                             0.9951                        1.0000                       1.0000            1.0000                        0.9166                        0.9242   
200                             0.9996                        1.0000                       1.0000            1.0000                        0.9233                        0.9271   
300                             1.0000                        1.0000                       1.0000            1.0000                        0.9263                        0.9288   
400                             1.0000                        1.0000                       1.0000            1.0000                        0.9284                        0.9300   

                                                         
Модель   HistGradientBoosting, lr=0.2 LightGBM, lr=0.05  
Итераций                                                 
10                             0.9036            0.9045  
25                             0.9171            0.9144  
50                             0.9183            0.9191  
100                            0.9285            0.9221  
200                            0.9277            0.9258  
300                            0.9282            0.9284  
400                            0.9282            0.9314

**Что видно по бустингу.**

* Классический `GradientBoostingClassifier` с параметрами по умолчанию (100 деревьев глубины 3, один поток) даёт 0.9051 за 23 с на этой машине (74 с в Colab в первой попытке Задания 8). Гистограммный `HistGradientBoostingClassifier` на тех же исходных признаках — 0.920 за 5 с, LightGBM — 0.923 за 3 с: при том же качестве или лучше они обучают в 4 раза больше деревьев за меньшее время.
* Агрегаты дают бустингу примерно +0.01 при одинаковых гиперпараметрах: HistGradientBoosting 0.920 → 0.930, LightGBM 0.923 → 0.931.
* Кривые обучения. При `learning_rate=0.02` качество ещё растёт на 400-й итерации (0.928) — за отведённое число шагов модель недообучена. При 0.05 кривая выходит на плато к 300–400 итерациям (0.9288 → 0.9300, разница в пределах шума). При 0.2 максимум 0.929 достигается уже примерно на 100-й итерации, после чего качество медленно снижается (0.928 к 400-й) — это переобучение: ROC-AUC на train равен 1.000 уже с 25-й итерации. Шаг обучения и число итераций взаимозаменяемы: маленький шаг требует больше деревьев, большой — раньше начинает переобучаться, а лучшие точки разных кривых различаются всего на 0.001–0.002.
* Число листьев: при 7 листьях (0.920–0.923) модели не хватает сложности, 15–31 лист (0.926–0.931) — оптимум для 2 520 строк. Все конфигурации бустинга на расширенных признаках лежат в узком диапазоне 0.920–0.931, то есть результат мало чувствителен к точным значениям гиперпараметров.
* ROC-AUC на train у всех моделей бустинга равен 1.000: как и лес, бустинг полностью разделяет обучающую выборку, но при умеренном шаге обучения это не мешает обобщению.

### 8.6. Референсные модели

Текст Задания 8 требует включить в сравнение логистическую регрессию. Приводятся два варианта: ровно та конфигурация, что была в первой попытке (`LogisticRegression(max_iter=1000)` на исходных признаках), и более аккуратная — со стандартизацией признаков и сильной L2-регуляризацией (`C=0.01`), на расширенном наборе. Второй ориентир — оценка «минус L1-сумма» из раздела 8.1, которая вообще не требует обучения.

In [29]:
run_experiment(
    "Логистическая регрессия (max_iter=1000) — из первой попытки Задания 8",
    RAW,
    lambda seed: LogisticRegression(max_iter=1000, random_state=seed),
    candidate=False,
)
run_experiment(
    "Логистическая регрессия (StandardScaler, C=0.01)",
    ENGINEERED,
    lambda seed: make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=5000, random_state=seed)),
    candidate=False,
)
experiments.append(
    {
        "Модель": "Минус L1-сумма расстояний (без обучения)",
        "Признаки": "1 агрегат (L1-сумма)",
        "Статус": REFERENCE,
        "ROC-AUC (train)": l1_train_auc,
        "ROC-AUC (валидация)": l1_validation_auc,
        "Время обучения, с": 0.0,
        "Время предсказания, с": 0.0,
        "seed": SEED,
    }
)
validation_predictions[("Минус L1-сумма расстояний (без обучения)", "1 агрегат (L1-сумма)")] = -X_validation.sum(axis=1)

0.8903 (train 0.9807,    1.7 с)  Логистическая регрессия (max_iter=1000) — из первой попытки Задания 8 — 512 исходных


0.9175 (train 0.9698,    0.5 с)  Логистическая регрессия (StandardScaler, C=0.01) — 512 исходных + 30 агрегатов


#### Опорные векторы с RBF-ядром — референс вне класса моделей задания

Деревья делят пространство признаков порогами, параллельными осям: чтобы описать близость двух *профилей расстояний* целиком, им нужно много разбиений, а каждое разбиение оценивается по части из 2 520 строк. Ядро RBF `exp(−γ‖u − v‖²)` сравнивает объекты сразу по всем координатам — через евклидово расстояние между стандартизованными векторами — и потому естественно описывает гладкую геометрию задачи: пары с похожими профилями расстояний получают похожие оценки без разбиения пространства на прямоугольники. Признаки для SVM — квадратные корни из 512 расстояний (сжимают тяжёлый правый хвост) и те же 30 агрегатов, всё стандартизовано.

Параметр `C` выбирается **5-кратной кросс-валидацией внутри train** (валидация при выборе не участвует), `gamma="scale"`. Качество считается по `decision_function` — ROC-AUC зависит только от порядка оценок, и калиброванные вероятности не нужны. Модель добавлена как ориентир: задание требует деревья и ансамбли, поэтому она не может стать итоговой.

In [30]:
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC


def svm_inputs(X_raw, X_fe):
    return np.hstack([np.sqrt(X_raw), X_fe[:, len(feature_columns):]])


def make_svm(C):
    return make_pipeline(StandardScaler(), SVC(C=C, kernel="rbf", gamma="scale"))


S_train = svm_inputs(X_train, X_train_fe)
S_validation = svm_inputs(X_validation, X_validation_fe)
S_test = svm_inputs(X_test, X_test_fe)
SVM_FEATURES = "sqrt(512 расстояний) + 30 агрегатов"
FEATURE_SETS[SVM_FEATURES] = (S_train, S_validation, S_test, None)

cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rows = []
for C in (1, 3, 10):
    started = time.perf_counter()
    oof = np.zeros(len(y_train))
    for fit_idx, score_idx in cv_folds.split(S_train, y_train):
        oof[score_idx] = make_svm(C).fit(S_train[fit_idx], y_train[fit_idx]).decision_function(S_train[score_idx])
    cv_rows.append({"C": C, "ROC-AUC (5-fold CV на train)": roc_auc_score(y_train, oof), "Время CV, с": time.perf_counter() - started})
svm_cv_table = pd.DataFrame(cv_rows)
display(svm_cv_table.round(4))
SVM_C = int(svm_cv_table.loc[svm_cv_table["ROC-AUC (5-fold CV на train)"].idxmax(), "C"])
print(f"Выбрано по кросс-валидации внутри train: C={SVM_C}")

SVM_NAME = f"SVM с RBF-ядром (C={SVM_C}, gamma=scale)"
_ = run_experiment(SVM_NAME, SVM_FEATURES, lambda seed: make_svm(SVM_C), candidate=False)

,C,ROC-AUC (5-fold CV на train),"Время CV, с"
0,1,0.9238,2.5129
1,3,0.9272,3.3554
2,10,0.9260,3.3551


Выбрано по кросс-валидации внутри train: C=3


0.9481 (train 1.0000,    0.8 с)  SVM с RBF-ядром (C=3, gamma=scale) — sqrt(512 расстояний) + 30 агрегатов


**Что показал референс.** Кросс-валидация внутри train выбирает `C=3` (0.9272 против 0.9238 при `C=1` и 0.9260 при `C=10`), и эта же модель даёт **0.948** на валидации — примерно на 0.015–0.018 больше лучших ансамблей деревьев на тех же 542 признаках (LightGBM 0.931, HistGradientBoosting 0.930), при времени обучения около 1 с. Парный бутстрэп в разделе 8.7 подтверждает, что эта разница не объясняется шумом валидации. Ограничения: на train SVM почти безошибочна (ROC-AUC 1.0), кросс-валидационная оценка 0.927 заметно ниже валидационной 0.948 — валидационная выборка, по-видимому, «легче» обучающей, и абсолютные значения на ней не стоит переносить на тест. Вывод для деревьев: их плато 0.925–0.935 — это не предел информации в данных, а ограничение осепараллельных разбиений на 2 520 строках. Следующий раздел передаёт деревьям недостающую информацию в виде одного признака.

### 8.7. Стекинг: оценка референсной модели как признак для деревьев

Если SVM видит в данных структуру, недоступную деревьям, её оценку можно передать деревьям как ещё один признак — это стекинг. Чтобы признак на обучающих строках не был «подсмотренным» (SVM, обученная на всём train, на тех же строках почти безошибочна — ROC-AUC 1.0), для train используются **вневыборочные (out-of-fold) оценки**: train делится на 5 стратифицированных фолдов, и оценка каждой строки получается от SVM, обученной на остальных четырёх. Для валидации и теста используется SVM, обученная на всём train. Валидация по-прежнему не участвует в обучении ни одной модели.

К 542 инженерным признакам добавляется один столбец — оценка SVM. На таких признаках обучаются те же ансамбли деревьев, что и раньше, а для честного сравнения рядом обучаются те же Extra Trees без этого столбца.

In [31]:
class StackedScore:
    def __init__(self, make_model, n_splits=5, seed=SEED):
        self.make_model = make_model
        self.n_splits = n_splits
        self.seed = seed

    def fit(self, X, y):
        folds = StratifiedKFold(n_splits=self.n_splits, shuffle=True, random_state=self.seed)
        self.oof_ = np.zeros(len(y))
        for fit_idx, score_idx in folds.split(X, y):
            self.oof_[score_idx] = self.make_model().fit(X[fit_idx], y[fit_idx]).decision_function(X[score_idx])
        self.model_ = self.make_model().fit(X, y)
        return self

    def transform(self, X):
        return self.model_.decision_function(X)


started = time.perf_counter()
svm_stack = StackedScore(lambda: make_svm(SVM_C)).fit(S_train, y_train)
print(
    f"Стекинг-оценка SVM: вневыборочный ROC-AUC на train {roc_auc_score(y_train, svm_stack.oof_):.4f}, "
    f"на валидации {roc_auc_score(y_validation, svm_stack.transform(S_validation)):.4f} "
    f"({time.perf_counter() - started:.1f} с)"
)

X_train_stack = np.column_stack([X_train_fe, svm_stack.oof_])
X_validation_stack = np.column_stack([X_validation_fe, svm_stack.transform(S_validation)])
X_test_stack = np.column_stack([X_test_fe, svm_stack.transform(S_test)])
stacked_columns = engineered_columns + ["stack_svm_score"]
STACKED = "542 инженерных + оценка SVM"
FEATURE_SETS[STACKED] = (X_train_stack, X_validation_stack, X_test_stack, stacked_columns)
assert X_train_stack.shape == (2520, 543) and X_validation_stack.shape == (560, 543) and X_test_stack.shape == (1116, 543)


def make_extra_trees(max_features):
    return lambda seed: ExtraTreesClassifier(n_estimators=800, max_features=max_features, random_state=seed, n_jobs=-1)


for max_features in (0.2, 0.3):
    name = f"Extra Trees (800 деревьев, max_features={max_features})"
    run_experiment(name, ENGINEERED, make_extra_trees(max_features))
    run_experiment(name, STACKED, make_extra_trees(max_features))
run_experiment(
    "Случайный лес (800 деревьев, max_features=0.2)",
    STACKED,
    lambda seed: RandomForestClassifier(n_estimators=800, max_features=0.2, random_state=seed, n_jobs=-1),
)
run_experiment(HGB_BEST, STACKED, make_hgb(0.05, 400, 31))
_ = run_experiment(LGBM_BEST, STACKED, make_lgbm(0.05, 400, 31, 0.5))

Стекинг-оценка SVM: вневыборочный ROC-AUC на train 0.9272, на валидации 0.9481 (4.6 с)


0.9292 (train 1.0000,    3.7 с)  Extra Trees (800 деревьев, max_features=0.2) — 512 исходных + 30 агрегатов


0.9432 (train 1.0000,    3.7 с)  Extra Trees (800 деревьев, max_features=0.2) — 542 инженерных + оценка SVM


0.9294 (train 1.0000,    5.4 с)  Extra Trees (800 деревьев, max_features=0.3) — 512 исходных + 30 агрегатов


0.9463 (train 1.0000,    5.3 с)  Extra Trees (800 деревьев, max_features=0.3) — 542 инженерных + оценка SVM


0.9455 (train 1.0000,   27.1 с)  Случайный лес (800 деревьев, max_features=0.2) — 542 инженерных + оценка SVM


0.9459 (train 1.0000,    4.9 с)  HistGradientBoosting (lr=0.05, 400 итераций, 31 лист) — 542 инженерных + оценка SVM


0.9432 (train 1.0000,    2.7 с)  LightGBM (lr=0.05, 400 итераций, 31 лист, colsample=0.5) — 542 инженерных + оценка SVM


In [32]:
et_stacked_auc = {
    mf: validation_predictions[(f"Extra Trees (800 деревьев, max_features={mf})", STACKED)] for mf in (0.2, 0.3)
}
et_stacked_auc = {mf: roc_auc_score(y_validation, p) for mf, p in et_stacked_auc.items()}
FINAL_MAX_FEATURES = max(et_stacked_auc, key=et_stacked_auc.get)
ET_STACK_BEST = f"Extra Trees (800 деревьев, max_features={FINAL_MAX_FEATURES})"
print("Extra Trees на стекинг-признаках:", {mf: round(a, 4) for mf, a in et_stacked_auc.items()}, "→ лучше", FINAL_MAX_FEATURES)


def rank_average(*scores):
    return np.mean([pd.Series(s).rank().to_numpy() / len(s) for s in scores], axis=0)


HYBRID_NAME = "Ранговое усреднение: Extra Trees (стекинг) + SVM"
HYBRID_FEATURES = f"{STACKED}; {SVM_FEATURES}"
hybrid_validation = rank_average(
    validation_predictions[(ET_STACK_BEST, STACKED)], validation_predictions[(SVM_NAME, SVM_FEATURES)]
)
hybrid_train = rank_average(
    fitted_models[(ET_STACK_BEST, STACKED)].predict_proba(X_train_stack)[:, 1],
    fitted_models[(SVM_NAME, SVM_FEATURES)].decision_function(S_train),
)
experiments.append(
    {
        "Модель": HYBRID_NAME,
        "Признаки": HYBRID_FEATURES,
        "Статус": REFERENCE,
        "ROC-AUC (train)": roc_auc_score(y_train, hybrid_train),
        "ROC-AUC (валидация)": roc_auc_score(y_validation, hybrid_validation),
        "Время обучения, с": sum(
            r["Время обучения, с"]
            for r in experiments
            if (r["Модель"], r["Признаки"]) in {(ET_STACK_BEST, STACKED), (SVM_NAME, SVM_FEATURES)}
        ),
        "Время предсказания, с": sum(
            r["Время предсказания, с"]
            for r in experiments
            if (r["Модель"], r["Признаки"]) in {(ET_STACK_BEST, STACKED), (SVM_NAME, SVM_FEATURES)}
        ),
        "seed": SEED,
    }
)
validation_predictions[(HYBRID_NAME, HYBRID_FEATURES)] = hybrid_validation
print(f"{HYBRID_NAME}: ROC-AUC на валидации {roc_auc_score(y_validation, hybrid_validation):.4f}")

Extra Trees на стекинг-признаках: {0.2: 0.9432, 0.3: 0.9463} → лучше 0.3


Ранговое усреднение: Extra Trees (стекинг) + SVM: ROC-AUC на валидации 0.9492


#### Парный бутстрэп по валидационным строкам

Оценки разных моделей получены на одних и тех же 560 строках, поэтому их разницу правильно оценивать **парно**: одни и те же 2 000 бутстрэп-выборок строк применяются к обеим моделям, и для каждой считается разница ROC-AUC. Так исключается общий для обеих моделей шум выборки. Ниже — средняя разница, 95 % интервал и доля выборок, где разница не положительна.

In [33]:
bootstrap_generator = np.random.default_rng(SEED)
bootstrap_indices = bootstrap_generator.integers(0, len(y_validation), size=(2000, len(y_validation)))


def bootstrap_aucs(scores):
    return np.array([roc_auc_score(y_validation[idx], scores[idx]) for idx in bootstrap_indices])


bootstrap_cache = {}


def paired_bootstrap(key_a, key_b):
    for key in (key_a, key_b):
        if key not in bootstrap_cache:
            bootstrap_cache[key] = bootstrap_aucs(validation_predictions[key])
    difference = bootstrap_cache[key_a] - bootstrap_cache[key_b]
    return {
        "Сравнение": f"{key_a[0]} [{key_a[1]}] − {key_b[0]} [{key_b[1]}]",
        "Средняя разница ROC-AUC": difference.mean(),
        "95 % интервал: от": np.quantile(difference, 0.025),
        "95 % интервал: до": np.quantile(difference, 0.975),
        "Доля выборок с разницей ≤ 0": (difference <= 0).mean(),
    }


ET_PLAIN_02 = ("Extra Trees (800 деревьев, max_features=0.2)", ENGINEERED)
ET_PLAIN_03 = ("Extra Trees (800 деревьев, max_features=0.3)", ENGINEERED)
paired_table = pd.DataFrame(
    [
        paired_bootstrap((SVM_NAME, SVM_FEATURES), (LGBM_BEST, ENGINEERED)),
        paired_bootstrap((SVM_NAME, SVM_FEATURES), (HGB_BEST, ENGINEERED)),
        paired_bootstrap(("Extra Trees (800 деревьев, max_features=0.3)", STACKED), ET_PLAIN_03),
        paired_bootstrap(("Extra Trees (800 деревьев, max_features=0.2)", STACKED), ET_PLAIN_02),
        paired_bootstrap((ET_STACK_BEST, STACKED), (HGB_BEST, ENGINEERED)),
        paired_bootstrap((ET_STACK_BEST, STACKED), (SVM_NAME, SVM_FEATURES)),
        paired_bootstrap((HYBRID_NAME, HYBRID_FEATURES), (SVM_NAME, SVM_FEATURES)),
    ]
)
display(paired_table.round(4))

,Сравнение,Средняя разница ROC-AUC,95 % интервал: от,95 % интервал: до,Доля выборок с разницей ≤ 0
0,"SVM с RBF-ядром (C=3, gamma=scale) [sqrt(512 расстояний) + 30 агрегатов] − LightGBM (lr=0.05, 400 итераций, 31 лист, colsample=0.5) [512 исходных + 30 агрегатов]",0.0167,0.0063,0.0278,0.0000
1,"SVM с RBF-ядром (C=3, gamma=scale) [sqrt(512 расстояний) + 30 агрегатов] − HistGradientBoosting (lr=0.05, 400 итераций, 31 лист) [512 исходных + 30 агрегатов]",0.0180,0.0069,0.0297,0.0005
2,"Extra Trees (800 деревьев, max_features=0.3) [542 инженерных + оценка SVM] − Extra Trees (800 деревьев, max_features=0.3) [512 исходных + 30 агрегатов]",0.0169,0.0102,0.0243,0.0000
3,"Extra Trees (800 деревьев, max_features=0.2) [542 инженерных + оценка SVM] − Extra Trees (800 деревьев, max_features=0.2) [512 исходных + 30 агрегатов]",0.0139,0.0075,0.0205,0.0000
4,"Extra Trees (800 деревьев, max_features=0.3) [542 инженерных + оценка SVM] − HistGradientBoosting (lr=0.05, 400 итераций, 31 лист) [512 исходных + 30 агрегатов]",0.0162,0.0083,0.0252,0.0000
5,"Extra Trees (800 деревьев, max_features=0.3) [542 инженерных + оценка SVM] − SVM с RBF-ядром (C=3, gamma=scale) [sqrt(512 расстояний) + 30 агрегатов]",-0.0018,-0.0089,0.0053,0.6870
6,"Ранговое усреднение: Extra Trees (стекинг) + SVM [542 инженерных + оценка SVM; sqrt(512 расстояний) + 30 агрегатов] − SVM с RBF-ядром (C=3, gamma=scale) [sqrt(512 расстояний) + 30 агрегатов]",0.0011,-0.0027,0.0049,0.2930


**Что дал стекинг.**

* Один столбец с вневыборочной оценкой SVM поднимает все ансамбли деревьев с плато 0.929–0.931 на новое плато 0.943–0.946: Extra Trees 800 деревьев `max_features=0.3` — с 0.929 до **0.946**, `max_features=0.2` — с 0.929 до 0.943, HistGradientBoosting — до 0.946, случайный лес — до 0.946 (но за 28 с), LightGBM — до 0.943. Вневыборочная оценка на train (0.927) честно отражает, что признак не «подсмотрен»: на обучающих строках он такой же зашумлённый, как будет на новых.
* Парный бутстрэп: разница «Extra Trees со стекингом − те же Extra Trees без него» равна +0.0169 с 95 % интервалом [+0.010; +0.024] для `max_features=0.3` и +0.0139 [+0.008; +0.021] для `max_features=0.2`; ни в одной из 2 000 бутстрэп-выборок разница не была отрицательной. Это статистически надёжный выигрыш, в отличие от разниц между конфигурациями деревьев внутри одного плато. Выигрыш над прежним лучшим кандидатом (HistGradientBoosting на 542 признаках) — +0.0162 [+0.008; +0.025].
* Деревья со стекингом **не превосходят саму SVM**: разница «Extra Trees (стекинг) − SVM» равна −0.0018 с интервалом [−0.009; +0.005], а ранговое усреднение Extra Trees и SVM (0.949) отличается от SVM на +0.0011 [−0.003; +0.005] — то есть не отличается. Гибрид и SVM остаются референсами: они вне класса моделей задания.
* Честный итог: ансамбль деревьев со стекингом — около 0.946 на валидации, SVM и гибрид — около 0.948–0.949. Бутстрэп-ошибка каждой из этих оценок около 0.008–0.011, поэтому значение 0.95 попадает в доверительный интервал, но утверждать «0.95» по 560 валидационным строкам нельзя.

### 8.8. Устойчивость к seed и усреднение моделей

Оценка на 560 валидационных строках сама по себе шумная, но у стохастических моделей есть и второй источник разброса — случайность обучения (бутстрэп, подвыборки признаков, случайные пороги). Для одного дерева без ограничений, случайного леса, Extra Trees, лучших моделей бустинга, усреднения моделей и Extra Trees на стекинг-признаках (итоговый кандидат) считаем ROC-AUC на валидации при пяти значениях seed (`SEED + 0 … SEED + 4`) и приводим среднее ± стандартное отклонение. `HistGradientBoostingClassifier` на выборке такого размера не использует случайность вообще (подвыборка при биннинге включается только при более чем 200 000 строк, ранняя остановка выключена), поэтому его разброс по seed равен нулю по построению.

In [34]:
seeds = [SEED + i for i in range(5)]
ET_BEST = "Extra Trees (400 деревьев, max_features=0.2)"
RF_REF = "Случайный лес (400 деревьев, max_features=sqrt)"
DEEP_TREE = "Дерево решений без ограничений (max_depth=None, min_samples_leaf=1)"
stability_candidates = {
    (DEEP_TREE, ENGINEERED): lambda seed: DecisionTreeClassifier(random_state=seed),
    (RF_REF, ENGINEERED): lambda seed: RandomForestClassifier(n_estimators=400, random_state=seed, n_jobs=-1),
    (ET_BEST, ENGINEERED): lambda seed: ExtraTreesClassifier(n_estimators=400, max_features=0.2, random_state=seed, n_jobs=-1),
    (HGB_BEST, ENGINEERED): make_hgb(0.05, 400, 31),
    (LGBM_BEST, ENGINEERED): make_lgbm(0.05, 400, 31, 0.5),
    (ET_STACK_BEST, STACKED): make_extra_trees(FINAL_MAX_FEATURES),
}
BLEND_NAME = "Усреднение вероятностей: HistGradientBoosting + LightGBM + Extra Trees"
blend_members = [HGB_BEST, LGBM_BEST, ET_BEST]

seed_predictions = {key: [] for key in stability_candidates}
started = time.perf_counter()
for (name, features), make_model in stability_candidates.items():
    for seed in seeds:
        if seed == SEED:
            seed_predictions[(name, features)].append(validation_predictions[(name, features)])
        else:
            _, _, proba = run_experiment(name, features, make_model, seed=seed, record=False)
            seed_predictions[(name, features)].append(proba)
seed_predictions[(BLEND_NAME, ENGINEERED)] = [
    np.mean([seed_predictions[(name, ENGINEERED)][i] for name in blend_members], axis=0) for i in range(len(seeds))
]
print(f"Обучение по пяти seed заняло {time.perf_counter() - started:.1f} с")

stability_rows = []
for (name, features), predictions in seed_predictions.items():
    aucs = np.array([roc_auc_score(y_validation, p) for p in predictions])
    stability_rows.append(
        {
            "Модель": name,
            "Признаки": features,
            "ROC-AUC по seed": ", ".join(f"{a:.4f}" for a in aucs),
            "Среднее": aucs.mean(),
            "std по 5 seed": aucs.std(),
        }
    )
stability_table = pd.DataFrame(stability_rows)
display(stability_table.round(4))
seed_std = {
    (name, features): std
    for name, features, std in zip(stability_table["Модель"], stability_table["Признаки"], stability_table["std по 5 seed"])
}

Обучение по пяти seed заняло 79.1 с


,Модель,Признаки,ROC-AUC по seed,Среднее,std по 5 seed
0,"Дерево решений без ограничений (max_depth=None, min_samples_leaf=1)",512 исходных + 30 агрегатов,"0.7339, 0.7375, 0.7446, 0.7589, 0.7464",0.7443,0.0086
1,"Случайный лес (400 деревьев, max_features=sqrt)",512 исходных + 30 агрегатов,"0.9179, 0.9173, 0.9187, 0.9182, 0.9183",0.9181,0.0005
2,"Extra Trees (400 деревьев, max_features=0.2)",512 исходных + 30 агрегатов,"0.9281, 0.9298, 0.9289, 0.9301, 0.9298",0.9293,0.0007
3,"HistGradientBoosting (lr=0.05, 400 итераций, 31 лист)",512 исходных + 30 агрегатов,"0.9300, 0.9300, 0.9300, 0.9300, 0.9300",0.9300,0.0000
4,"LightGBM (lr=0.05, 400 итераций, 31 лист, colsample=0.5)",512 исходных + 30 агрегатов,"0.9314, 0.9286, 0.9337, 0.9318, 0.9297",0.9310,0.0018
5,"Extra Trees (800 деревьев, max_features=0.3)",542 инженерных + оценка SVM,"0.9463, 0.9465, 0.9456, 0.9453, 0.9469",0.9461,0.0006
6,Усреднение вероятностей: HistGradientBoosting + LightGBM + Extra Trees,512 исходных + 30 агрегатов,"0.9330, 0.9320, 0.9335, 0.9342, 0.9322",0.9330,0.0008


In [35]:
blend_probability = np.mean([validation_predictions[(name, ENGINEERED)] for name in blend_members], axis=0)
experiments.append(
    {
        "Модель": BLEND_NAME,
        "Признаки": ENGINEERED,
        "Статус": CANDIDATE,
        "ROC-AUC (train)": roc_auc_score(
            y_train,
            np.mean([fitted_models[(name, ENGINEERED)].predict_proba(X_train_fe)[:, 1] for name in blend_members], axis=0),
        ),
        "ROC-AUC (валидация)": roc_auc_score(y_validation, blend_probability),
        "Время обучения, с": sum(
            r["Время обучения, с"] for r in experiments if r["Модель"] in blend_members and r["Признаки"] == ENGINEERED
        ),
        "Время предсказания, с": sum(
            r["Время предсказания, с"] for r in experiments if r["Модель"] in blend_members and r["Признаки"] == ENGINEERED
        ),
        "seed": SEED,
    }
)
validation_predictions[(BLEND_NAME, ENGINEERED)] = blend_probability
print(f"{BLEND_NAME}: ROC-AUC на валидации {roc_auc_score(y_validation, blend_probability):.4f}")

Усреднение вероятностей: HistGradientBoosting + LightGBM + Extra Trees: ROC-AUC на валидации 0.9330


**Устойчивость к seed.**

* Одно дерево без ограничений: 0.744 ± 0.009. При смене seed меняется только порядок перебора признаков (разрешение «ничьих» между одинаково хорошими разбиениями), но у глубокого дерева это меняет всю структуру и качество на валидации на ±0.01.
* Случайный лес и Extra Trees — 0.918 ± 0.0005 и 0.929 ± 0.0007: усреднение 400 деревьев делает результат почти нечувствительным к случайности, хотя каждое дерево внутри случайно. Это второе проявление того же снижения дисперсии.
* LightGBM — 0.931 ± 0.0018: подвыборки строк (`subsample=0.8`) и признаков (`colsample=0.5`) вносят случайность в каждую итерацию, а последовательное построение деревьев её накапливает, поэтому разброс больше, чем у леса. HistGradientBoosting детерминирован (std = 0).
* Усреднение вероятностей трёх моделей — 0.933 ± 0.0008: разброс меньше, чем у LightGBM, и близок к Extra Trees, но не ниже нуля у HistGradientBoosting.
* Extra Trees на стекинг-признаках (итоговый кандидат) — 0.946 ± 0.0006: добавленный столбец SVM не делает ансамбль менее устойчивым; разбиение на фолды для вневыборочных оценок при этом фиксировано (`random_state=SEED`).

Все эти разбросы (≤ 0.002) заметно меньше бутстрэп-ошибки самой валидационной оценки (около 0.009, см. 8.10): главный источник неопределённости — небольшая валидационная выборка из 560 строк, а не случайность обучения.

### 8.9. Сводная таблица экспериментов

Все строки получены на одном и том же разбиении train/validation; модели обучены только на train. Столбец «std по 5 seed» заполнен для моделей из проверки устойчивости (8.8); столбец «Статус» отмечает референсные модели вне класса моделей задания, которые не могут быть выбраны итоговыми. Таблица также сохраняется в `validation_results.csv`.

In [36]:
results_table = pd.DataFrame(experiments)
results_table["std по 5 seed"] = [
    seed_std.get((name, features), np.nan) for name, features in zip(results_table["Модель"], results_table["Признаки"])
]
results_table = results_table.sort_values("ROC-AUC (валидация)", ascending=False).reset_index(drop=True)
results_table = results_table.round(
    {"ROC-AUC (train)": 4, "ROC-AUC (валидация)": 4, "Время обучения, с": 2, "Время предсказания, с": 3, "std по 5 seed": 4}
)
results_table.to_csv(DATA_DIR / "validation_results.csv", index=False)
display(results_table)

,Модель,Признаки,Статус,ROC-AUC (train),ROC-AUC (валидация),"Время обучения, с","Время предсказания, с",seed,std по 5 seed
0,Ранговое усреднение: Extra Trees (стекинг) + SVM,542 инженерных + оценка SVM; sqrt(512 расстояний) + 30 агрегатов,"референс, вне класса моделей задания",1.0000,0.9492,6.11,0.292,20260916,NaN
1,"SVM с RBF-ядром (C=3, gamma=scale)",sqrt(512 расстояний) + 30 агрегатов,"референс, вне класса моделей задания",1.0000,0.9481,0.79,0.197,20260916,NaN
2,"Extra Trees (800 деревьев, max_features=0.3)",542 инженерных + оценка SVM,кандидат,1.0000,0.9463,5.32,0.095,20260916,0.0006
3,"HistGradientBoosting (lr=0.05, 400 итераций, 31 лист)",542 инженерных + оценка SVM,кандидат,1.0000,0.9459,4.93,0.007,20260916,NaN
4,"Случайный лес (800 деревьев, max_features=0.2)",542 инженерных + оценка SVM,кандидат,1.0000,0.9455,27.13,0.097,20260916,NaN
5,"LightGBM (lr=0.05, 400 итераций, 31 лист, colsample=0.5)",542 инженерных + оценка SVM,кандидат,1.0000,0.9432,2.74,0.005,20260916,NaN
6,"Extra Trees (800 деревьев, max_features=0.2)",542 инженерных + оценка SVM,кандидат,1.0000,0.9432,3.68,0.096,20260916,NaN
7,Усреднение вероятностей: HistGradientBoosting + LightGBM + Extra Trees,512 исходных + 30 агрегатов,кандидат,1.0000,0.9330,9.02,0.066,20260916,0.0008
8,"LightGBM (lr=0.05, 400 итераций, 31 лист, colsample=0.5)",512 исходных + 30 агрегатов,кандидат,1.0000,0.9314,2.54,0.004,20260916,0.0018
9,"LightGBM (lr=0.02, 1000 итераций, 15 листьев, colsample=0.3)",512 исходных + 30 агрегатов,кандидат,1.0000,0.9305,2.41,0.008,20260916,NaN


### 8.10. Выбор итоговой модели

Критерии: качество на валидации, устойчивость (разброс по seed и бутстрэп-ошибка самой валидационной оценки), время работы, интерпретируемость и воспроизводимость (число сторонних библиотек). Более сложная модель не считается лучшей автоматически: выигрыш должен быть заметен на фоне шума оценки — здесь это проверяется парным бутстрэпом из раздела 8.7. Итоговой может стать только модель со статусом «кандидат».

In [37]:
def bootstrap_auc_se(probability, n_bootstrap=1000, seed=SEED):
    generator = np.random.default_rng(seed)
    indices = generator.integers(0, len(y_validation), size=(n_bootstrap, len(y_validation)))
    aucs = [roc_auc_score(y_validation[idx], probability[idx]) for idx in indices]
    return float(np.std(aucs))


selection_view = results_table.set_index(["Модель", "Признаки"])
comparison_keys = [
    (ET_STACK_BEST, STACKED),
    ("Extra Trees (800 деревьев, max_features=0.2)" if FINAL_MAX_FEATURES == 0.3 else "Extra Trees (800 деревьев, max_features=0.3)", STACKED),
    ("Случайный лес (800 деревьев, max_features=0.2)", STACKED),
    (HGB_BEST, STACKED),
    (LGBM_BEST, STACKED),
    (ET_STACK_BEST, ENGINEERED),
    (HGB_BEST, ENGINEERED),
    (LGBM_BEST, ENGINEERED),
    (BLEND_NAME, ENGINEERED),
    (SVM_NAME, SVM_FEATURES),
    (HYBRID_NAME, HYBRID_FEATURES),
]
display(selection_view.loc[comparison_keys, ["Статус", "ROC-AUC (валидация)", "Время обучения, с", "std по 5 seed"]])

candidates_table = results_table[results_table["Статус"] == CANDIDATE].reset_index(drop=True)
best_candidate = candidates_table.iloc[0]
final_row = selection_view.loc[(ET_STACK_BEST, STACKED)]
print(f"Лучший кандидат по валидации: {best_candidate['Модель']} [{best_candidate['Признаки']}] — {best_candidate['ROC-AUC (валидация)']:.4f}")
print(f"Итоговый кандидат: {ET_STACK_BEST} [{STACKED}] — {final_row['ROC-AUC (валидация)']:.4f} "
      f"(разница с лучшим: {final_row['ROC-AUC (валидация)'] - best_candidate['ROC-AUC (валидация)']:+.4f})")
print(f"Бутстрэп-ошибка ROC-AUC итогового кандидата на валидации: {bootstrap_auc_se(validation_predictions[(ET_STACK_BEST, STACKED)]):.4f}")
print(f"Бутстрэп-ошибка ROC-AUC SVM на валидации: {bootstrap_auc_se(validation_predictions[(SVM_NAME, SVM_FEATURES)]):.4f}")

,,Статус,ROC-AUC (валидация),"Время обучения, с",std по 5 seed
Модель,Признаки,,,,
"Extra Trees (800 деревьев, max_features=0.3)",542 инженерных + оценка SVM,кандидат,0.9463,5.32,0.0006
"Extra Trees (800 деревьев, max_features=0.2)",542 инженерных + оценка SVM,кандидат,0.9432,3.68,NaN
"Случайный лес (800 деревьев, max_features=0.2)",542 инженерных + оценка SVM,кандидат,0.9455,27.13,NaN
"HistGradientBoosting (lr=0.05, 400 итераций, 31 лист)",542 инженерных + оценка SVM,кандидат,0.9459,4.93,NaN
"LightGBM (lr=0.05, 400 итераций, 31 лист, colsample=0.5)",542 инженерных + оценка SVM,кандидат,0.9432,2.74,NaN
"Extra Trees (800 деревьев, max_features=0.3)",512 исходных + 30 агрегатов,кандидат,0.9294,5.36,NaN
"HistGradientBoosting (lr=0.05, 400 итераций, 31 лист)",512 исходных + 30 агрегатов,кандидат,0.9300,4.66,0.0000
"LightGBM (lr=0.05, 400 итераций, 31 лист, colsample=0.5)",512 исходных + 30 агрегатов,кандидат,0.9314,2.54,0.0018
Усреднение вероятностей: HistGradientBoosting + LightGBM + Extra Trees,512 исходных + 30 агрегатов,кандидат,0.9330,9.02,0.0008


Лучший кандидат по валидации: Extra Trees (800 деревьев, max_features=0.3) [542 инженерных + оценка SVM] — 0.9463
Итоговый кандидат: Extra Trees (800 деревьев, max_features=0.3) [542 инженерных + оценка SVM] — 0.9463 (разница с лучшим: +0.0000)


Бутстрэп-ошибка ROC-AUC итогового кандидата на валидации: 0.0080


Бутстрэп-ошибка ROC-AUC SVM на валидации: 0.0081


**Решение: итоговая модель — `ExtraTreesClassifier(n_estimators=800, max_features=0.3, n_jobs=-1, random_state=SEED)` на 542 инженерных признаках + стекинг-оценке `SVC(C=3, kernel="rbf", gamma="scale")` после `StandardScaler` (вневыборочные оценки по 5 стратифицированным фолдам с `random_state=SEED`; остальные параметры Extra Trees по умолчанию: `criterion="gini"`, `min_samples_leaf=1`, `bootstrap=False`).** ROC-AUC на валидации 0.9463, по пяти seed 0.9461 ± 0.0006, бутстрэп-ошибка оценки 0.008.

Почему именно эта модель:

* **Выигрыш от стекинга статистически надёжен.** Те же Extra Trees без столбца SVM дают 0.929; парный бутстрэп разницы +0.0169 с 95 % интервалом [+0.010; +0.024] (раздел 8.7). Это единственное изменение в исследовании, которое сдвинуло качество деревьев за пределы шума валидации: все остальные варианты — гиперпараметры, усреднение моделей, другие ансамбли — оставались внутри плато 0.925–0.935.
* **Остаётся в классе моделей задания.** Предсказание делает ансамбль деревьев; SVM участвует только как источник одного признака, так же как агрегаты строки. Хуже ли это «чистого» ансамбля с точки зрения задания — вопрос трактовки, поэтому в таблице сохранены и оценки без стекинга (Extra Trees 0.929, HistGradientBoosting 0.930), и референсы вне класса.
* **Простая, быстрая и воспроизводимая.** Только scikit-learn, обучение около 10 с вместе с SVM, детерминированный результат при любом числе потоков, разброс по seed 0.0006 — не больше, чем у ансамблей без стекинга.
* **Соседи по таблице не лучше.** HistGradientBoosting (0.9459) и случайный лес (0.9455) на тех же стекинг-признаках отличаются от Extra Trees на 0.0004–0.0008 — в десять раз меньше ошибки оценки; выбрана самая быстрая и наименее чувствительная к гиперпараметрам из трёх (случайный лес при этом в пять раз медленнее). `max_features=0.3` против 0.2: 0.9463 против 0.9432 — разница тоже в пределах шума, выбор сделан по валидации, как и предусмотрено протоколом, и его оптимистичность учтена в оговорках ниже.
* **SVM и гибрид не выбраны**, хотя их числа выше на 0.002–0.003 (0.9481 и 0.9492): разница внутри шума (интервалы включают ноль), а сами модели вне класса моделей задания. Они остаются референсами в таблице.

Оговорки. Валидационная выборка — 560 строк, стандартная ошибка ROC-AUC около 0.008–0.011; выбор `max_features` и сравнение с соседями сделаны по этой же выборке, поэтому оценка 0.946 слегка оптимистична. Уверенно утверждать можно следующее: стекинг даёт ≈ +0.015 к деревьям, итоговое качество лежит в интервале примерно 0.93–0.96, и значение 0.95 этой выборкой не подтверждается и не опровергается. Относительно первой попытки Задания 8: там наилучшим был градиентный бустинг (0.9051), а не дерево решений (0.7473), как ошибочно было написано в выводе; здесь качество доведено до 0.946 при времени обучения в несколько раз меньше, чем у `GradientBoostingClassifier`.

### 8.11. Итоговое обучение и файл отправки

Итоговая модель обучается **только на `train.csv`** (объединять train и validation ноутбук запрещает): сначала SVM даёт вневыборочные оценки для train и оценки для validation/test, затем на 542 инженерных признаках и этом столбце обучаются Extra Trees. Тестовая выборка предсказывается дважды для проверки детерминированности, файл отправки собирается из `sample_submission` и проходит те же проверки, что и в Задании 7.

In [38]:
FINAL_MODEL_NAME = ET_STACK_BEST
FINAL_FEATURES = STACKED
started = time.perf_counter()
final_stack = StackedScore(lambda: make_svm(SVM_C)).fit(S_train, y_train)
X_train_final = np.column_stack([X_train_fe, final_stack.oof_])
X_validation_final = np.column_stack([X_validation_fe, final_stack.transform(S_validation)])
X_test_final = np.column_stack([X_test_fe, final_stack.transform(S_test)])
final_model = ExtraTreesClassifier(n_estimators=800, max_features=FINAL_MAX_FEATURES, n_jobs=-1, random_state=SEED)
print(final_stack.model_)
print(final_model)
final_model.fit(X_train_final, y_train)
final_fit_seconds = time.perf_counter() - started
final_validation_probability = final_model.predict_proba(X_validation_final)[:, 1]
final_validation_auc = roc_auc_score(y_validation, final_validation_probability)
assert np.isclose(final_validation_auc, selection_view.loc[(FINAL_MODEL_NAME, FINAL_FEATURES), "ROC-AUC (валидация)"], atol=1e-4)

final_test_probability = final_model.predict_proba(X_test_final)[:, 1]
final_test_probability_repeat = final_model.predict_proba(X_test_final)[:, 1]
assert final_test_probability.shape == (len(test),)
assert np.array_equal(final_test_probability, final_test_probability_repeat)
assert np.all((final_test_probability >= 0.0) & (final_test_probability <= 1.0))

submission = sample_submission.copy()
submission["target"] = final_test_probability
assert list(submission.columns) == ["row_id", "target"]
assert submission.shape == (1116, 2)
assert (submission["row_id"].to_numpy() == test["row_id"].to_numpy()).all()
assert (submission["row_id"].to_numpy() == sample_submission["row_id"].to_numpy()).all()
assert submission["row_id"].is_unique
assert not submission.isna().any().any()
assert submission["target"].between(0.0, 1.0).all()
submission.to_csv(DATA_DIR / "submission.csv", index=False)

print(f"Итоговая модель: {FINAL_MODEL_NAME} — {FINAL_FEATURES}")
print(f"ROC-AUC итоговой модели на валидации: {final_validation_auc:.6f} (базовое дерево: {validation_auc:.6f})")
print(f"Время обучения итоговой модели (SVM с вневыборочными оценками + Extra Trees): {final_fit_seconds:.1f} с")
print(f"Средняя предсказанная вероятность на тесте: {final_test_probability.mean():.3f}")
print(submission.head())
print(f"Время работы Задания 8: {time.perf_counter() - study_started:.1f} с")
print(f"Общее время работы ноутбука: {time.perf_counter() - notebook_started:.1f} с")

Pipeline(steps=[('standardscaler', StandardScaler()), ('svc', SVC(C=3))])
ExtraTreesClassifier(max_features=0.3, n_estimators=800, n_jobs=-1,
                     random_state=20260916)


Итоговая модель: Extra Trees (800 деревьев, max_features=0.3) — 542 инженерных + оценка SVM
ROC-AUC итоговой модели на валидации: 0.946327 (базовое дерево: 0.747309)
Время обучения итоговой модели (SVM с вневыборочными оценками + Extra Trees): 10.5 с
Средняя предсказанная вероятность на тесте: 0.504
                     row_id  target
0  row_46ad1afad8562239f79c  0.9250
1  row_b42ade9cd57b8d919acb  0.9487
2  row_c951cd5e197b6b4e8b83  0.2712
3  row_a54b3423760d599416ce  0.0512
4  row_a87c87cc47690e2a303f  0.9838
Время работы Задания 8: 333.0 с
Общее время работы ноутбука: 333.7 с


### 8.12. Воспроизводимость

* `SEED = 20260916` задаёт `random.seed`, `np.random.seed`, все `random_state` и генераторы бутстрэпа; проверка устойчивости использует фиксированные `SEED + 0 … SEED + 4`.
* Итоговая модель — `SVC` (libsvm, детерминированный алгоритм без случайности при `probability=False`) со стратифицированными фолдами `random_state=SEED` и `ExtraTreesClassifier` с `random_state=SEED`: в scikit-learn результат леса не зависит от `n_jobs`, поэтому «Restart kernel → Run all» воспроизводит `submission.csv` байт в байт при любом числе потоков.
* LightGBM участвует только в сравнении; его результаты повторяются при `deterministic=True`, `force_row_wise=True` и **том же числе потоков** (`n_jobs=4`): при другом числе потоков порядок суммирования гистограмм меняется, и значения ROC-AUC могут отличаться в последних знаках.
* Сторонние библиотеки: numpy, pandas, scikit-learn, lightgbm. Версии — ниже.

In [39]:
import sklearn

print("Python:", sys.version)
for module in (np, pd, sklearn, lgb):
    print(f"{module.__name__}: {module.__version__}")

Python: 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
numpy: 2.4.4
pandas: 3.0.6
sklearn: 1.9.1
lightgbm: 4.7.0


## 10. Как использовать результат и загрузить на Kaggle

1. **Получить файл отправки.** Откройте ноутбук в Jupyter рядом с четырьмя CSV-файлами и выполните «Kernel → Restart kernel and Run all cells». Через несколько минут рядом с ноутбуком появится `submission.csv` — 1 116 строк, два столбца `row_id,target`, в `target` — вероятность класса 1. Тот же файл получается командой `jupyter nbconvert --to notebook --execute --inplace solution_student_ru.ipynb`; повторный запуск даёт байт в байт тот же файл.
2. **Загрузка через сайт Kaggle.** На странице соревнования нажмите «Submit Predictions», перетащите `submission.csv`, при желании добавьте описание (например, «Extra Trees + SVM stack, val AUC 0.946») и нажмите «Submit». Публичный результат появится в разделе «My Submissions» / «Submissions». Если правила соревнования требуют выбрать итоговые посылки, отметьте их там до дедлайна — иначе будут взяты последние по умолчанию.
3. **Загрузка через Kaggle API.** Установите клиент `pip install kaggle`, создайте токен на сайте (Account → API → Create New Token) и положите скачанный `kaggle.json` в `~/.kaggle/` (`chmod 600 ~/.kaggle/kaggle.json`) или в `C:\Users\<user>\.kaggle\`. Затем:

```bash
kaggle competitions submit -c <competition-slug> -f submission.csv -m "ExtraTrees + SVM stack, val AUC 0.946"
kaggle competitions submissions -c <competition-slug>
```

   `<competition-slug>` — идентификатор соревнования из адреса его страницы (`kaggle.com/competitions/<competition-slug>`).
4. **Запуск внутри Kaggle Notebook.** Создайте ноутбук в соревновании (или загрузите этот файл через «File → Import Notebook»), через «Add Data» подключите данные соревнования и замените `DATA_DIR = Path(".")` на `DATA_DIR = Path("/kaggle/input/<competition-slug>")`; выходные файлы пишутся в `/kaggle/working/`, поэтому в ячейках сохранения укажите `Path("/kaggle/working") / "submission.csv"`. После «Save Version» (Save & Run All) файл можно отправить кнопкой «Submit to Competition» прямо из вкладки Output.
5. **Запуск в Google Colab.** Загрузите ноутбук и четыре CSV-файла в сессию (значок папки слева → Upload), выполните «Runtime → Run all», затем скачайте `submission.csv` из той же панели файлов и загрузите его на Kaggle любым способом выше.
6. **Предупреждение о лидерборде.** Публичный результат считается на части тестовой выборки и отличается от валидационного: у базового дерева 0.7473 на валидации против ориентировочных 0.7216 на публичном лидерборде. Публичный лидерборд нельзя использовать для выбора модели — это скрытая подгонка под тест; все решения в этом ноутбуке приняты по `validation.csv`.